# PHM 2016 Data Challenge (CMP) — Data Preprocessing, EDA & Insights

**Dataset:** [akangel0307/PHM-Data-Challenge](https://github.com/akangel0307/PHM-Data-Challenge) — a copy of the **2016 PHM Society Data Challenge** dataset on **Chemical-Mechanical Planarization (CMP)** of semiconductor wafers.

**What this notebook delivers** (in the recommended report order *objective → data understanding → initial audit → preprocessing decisions → final quality checks → EDA → insights → recommendations & limitations*):

| # | Section | # | Section |
|---|---|---|---|
| 0 | Setup, helpers, cleaning log | 13 | Categorical variables |
| 1 | Objective & unit of observation | 14 | Numerical variables |
| 2 | Sources & collection | 15 | Feature engineering (wafer-run level) |
| 3 | Raw data preservation & reproducibility | 16 | Target quality & distribution |
| 4 | Data dictionary & schema validation | 17 | Time-series specific checks |
| 5 | Initial data audit | 18 | Univariate EDA |
| 6 | Types, formats, representations | 19 | Bivariate & multivariate EDA |
| 7 | Value & cross-field validation | 20 | Segments, trends, comparisons |
| 8 | Missing data | 21 | Robustness & uncertainty |
| 9 | Duplicates | 22 | Insights (finding → evidence → meaning → caveat → next step) |
| 10 | Joins / integration (sensor ↔ labels) | 23 | Final validation, before/after |
| 11 | Outliers | 24 | Deliverables export |
| 12 | Leakage prevention & split strategy | | + leakage-safe baseline models |

> **How to use:** `Runtime → Run all`. Every cell prints what it found. The notebook is written to *discover* the repository layout at run time (it does not hard-code file names), because the repo contains several copies / processed versions of the data. Where a decision depends on what the data shows, the cell prints the evidence and the decision rule it applied, and logs it to the **cleaning log**.

## 0. Setup
Configuration, imports, plotting style, and two helpers used everywhere:
* `log_step(...)` — appends to the **cleaning / transformation log** (what changed, why, how many records, method).
* `save_fig(...)` — saves every figure to `outputs/figures/` so the report can reuse it.

In [ ]:
import os, sys, re, io, json, glob, hashlib, zipfile, shutil, warnings, urllib.request, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

IN_COLAB = "google.colab" in sys.modules
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------- configuration ----------------
DATA_URL = "https://github.com/akangel0307/PHM-Data-Challenge/archive/refs/heads/master.zip"
WORK_DIR = Path(os.environ.get("PHM_WORK_DIR", "/content/phm" if IN_COLAB else "phm_work"))
RAW_ZIP = WORK_DIR / "raw" / "PHM-Data-Challenge-master.zip"
EXTRACT_DIR = WORK_DIR / "raw" / "extracted"      # raw files - never modified
OUT_DIR = WORK_DIR / "outputs"                    # every derived artefact goes here
FIG_DIR = OUT_DIR / "figures"
for d in (RAW_ZIP.parent, EXTRACT_DIR, OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TARGET = "AVG_REMOVAL_RATE"
RUN_KEY = ["SPLIT", "WAFER_ID", "STAGE"]          # one polishing run = one labelled unit

# ---------------- plotting style ----------------
# Fixed-order categorical palette (colour-blind-validated reference palette; slots used in order, never cycled)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SPLIT_COLORS = {"train": PALETTE[0], "validation": PALETTE[1], "test": PALETTE[2]}
sns.set_theme(style="whitegrid", palette=PALETTE, rc={
    "axes.spines.top": False, "axes.spines.right": False, "grid.color": "#e6e6e3",
    "axes.edgecolor": "#b5b4ad", "lines.linewidth": 2, "figure.dpi": 110, "axes.titleweight": "bold"})

def save_fig(name, fig=None):
    fig = fig or plt.gcf()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

# ---------------- cleaning / transformation log ----------------
CLEAN_LOG = []
def log_step(step, action, reason, n_affected=None, method=""):
    CLEAN_LOG.append({"time": dt.datetime.now().isoformat(timespec="seconds"), "step": step,
                      "action": action, "reason": reason,
                      "records_affected": None if n_affected is None else int(n_affected),
                      "method": method})
    print(f"[LOG] {step}: {action}" + ("" if n_affected is None else f"  (records affected: {int(n_affected):,})"))

def md(text):
    display(Markdown(text))

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| Colab:", IN_COLAB)
print("Working directory:", WORK_DIR.resolve())

## 1. Objective & unit of observation

**Problem statement.** In CMP, a wafer is pressed against a rotating polishing pad while slurry is fed in, flattening the wafer surface. The amount of material removed per unit time — the **material removal rate (MRR)** — determines whether the wafer meets specification, but measuring it requires a slow offline metrology step. The PHM 2016 challenge asked: *can the average MRR of a polishing run be predicted from the machine's sensor and consumable-usage signals recorded during that run (virtual metrology)?*

**Units of observation — there are two, and mixing them is the most common mistake with this dataset:**

| Table | One row = | Key |
|---|---|---|
| Sensor files (`CMP-*-NNN.csv`) | one **time-stamped sensor sample** during the polishing of one wafer at one stage | `WAFER_ID` + `STAGE` + `TIMESTAMP` |
| Label files (`*removalrate*.csv`) | one **polishing run** (a wafer at a stage) with its measured average removal rate | `WAFER_ID` + `STAGE` |

Hundreds of sensor rows describe a single labelled run, so they are **not independent observations**. All statistical analysis of the target is therefore done at the **run level** after aggregating the time series (Section 15).

**Analysis questions**
1. Is the data complete, consistent and trustworthy enough to model? (quality)
2. What does a typical polishing run look like (duration, sampling, signal levels), and how much does it vary?
3. How is the removal rate distributed, are there distinct operating modes (stage / chamber), and are there invalid labels?
4. Which process variables (pressures, rotations, slurry flow) and **consumable wear** (usage counters of pad, dresser, membrane, …) are associated with removal rate?
5. Are the given train / validation / test splits comparable (no distribution shift, no leakage)?

**Intended use:** descriptive analysis **and** preparation for supervised regression (target = `AVG_REMOVAL_RATE`).
**Prediction time:** at the *end* of a polishing run, before metrology. Usable inputs = everything recorded up to the end of that run. Labels of *the same run* are not inputs; labels of other runs may only be used if they would truly be available (metrology of earlier wafers) — see Section 12.

**Key metric definitions**
* **MRR / `AVG_REMOVAL_RATE`** – average removal rate of the run (units not published by the organisers; values are on an anonymised scale).
* **Run duration** – last `TIMESTAMP` − first `TIMESTAMP` within the run.
* **Consumable age at start** – value of each `USAGE_OF_*` counter at the first sample of the run.

## 2. Data sources & collection

| Item | Detail |
|---|---|
| Original source | PHM Society 2016 Data Challenge (CMP wafer polishing), released for a competition |
| This copy | GitHub repo `akangel0307/PHM-Data-Challenge` (contains the original files **and** the author's processed versions / experiment folders) |
| Collection | Machine sensor logs recorded during polishing; removal rate measured offline by metrology |
| Splits | Organiser-defined **training**, **validation** and **test** sets |
| Units | Not published – values are anonymised / scaled. Interpret **relative** differences, not absolute physics |

**Known / likely limitations (checked below rather than assumed):**
* The repo may hold **duplicate copies** of the same files in different folders → de-duplicated by checksum (Section 3).
* Only the runs the organisers chose to release are included; we cannot tell whether they are a random sample of production.
* Machine identities, units and the exact meaning of some fields (e.g. `MACHINE_DATA`) are not documented.
* Usage counters are cumulative; their reset points (consumable replacement) are not explicitly recorded.

## 3. Preserve the raw data & make the workflow reproducible
* The zip is downloaded once and its **SHA-256** recorded.
* Files are extracted to `raw/extracted/` and **never modified**; all cleaning happens on in-memory copies.
* Nested archives (`.zip`) inside the repo are extracted next to themselves.
* A **file manifest** (path, size, MD5, detected role, split) is saved. Byte-identical files stored in several folders are detected by MD5 and loaded only once.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if not RAW_ZIP.exists():
    print("Downloading", DATA_URL)
    urllib.request.urlretrieve(DATA_URL, RAW_ZIP)
else:
    print("Using cached archive:", RAW_ZIP)

if not zipfile.is_zipfile(RAW_ZIP):
    raise RuntimeError("Download is not a valid zip. Download it manually from GitHub and upload it to " + str(RAW_ZIP))

ZIP_SHA256 = sha256(RAW_ZIP)
EXTRACTED_AT = dt.datetime.now().isoformat(timespec="seconds")
print(f"Archive size: {RAW_ZIP.stat().st_size/1e6:,.1f} MB | SHA-256: {ZIP_SHA256}")

if not any(EXTRACT_DIR.iterdir()):
    with zipfile.ZipFile(RAW_ZIP) as z:
        z.extractall(EXTRACT_DIR)

# Extract nested zip archives (up to 3 levels)
for level in range(3):
    nested = [p for p in EXTRACT_DIR.rglob("*.zip") if not (p.parent / (p.stem + "__unzipped")).exists()]
    if not nested:
        break
    for p in nested:
        try:
            with zipfile.ZipFile(p) as z:
                z.extractall(p.parent / (p.stem + "__unzipped"))
            print("Extracted nested archive:", p.relative_to(EXTRACT_DIR))
        except zipfile.BadZipFile:
            print("WARNING: could not open nested archive", p)

other_archives = [p for p in EXTRACT_DIR.rglob("*") if p.suffix.lower() in (".rar", ".7z", ".gz", ".tar")]
if other_archives:
    print("NOTE: archives not auto-extracted (extract manually if they hold data):", [str(p.relative_to(EXTRACT_DIR)) for p in other_archives])

### 3.1 File inventory & role detection
Each CSV header is read and the file is classified by its **columns**, not its name:
* **sensor** – has `WAFER_ID` and `TIMESTAMP`
* **label** – has `WAFER_ID` and `AVG_REMOVAL_RATE`
* **label** must contain *only* `WAFER_ID`, `STAGE`, `AVG_REMOVAL_RATE` – the repo also holds processed feature tables (e.g. `data_final.csv`, 100+ columns) that include the target; those are **other**
* **other** – processed tables, experiment outputs (kept in the manifest, not used as raw input); `__MACOSX` / `._*` files are OS metadata and ignored

The split (train / validation / test) is inferred from the file path.

In [ ]:
def infer_split(path_str):
    s = path_str.lower()
    if "train" in s: return "train"
    if "valid" in s: return "validation"
    if "test" in s:  return "test"
    return "unknown"

rows = []
for p in sorted(EXTRACT_DIR.rglob("*")):
    if not p.is_file():
        continue
    rel = str(p.relative_to(EXTRACT_DIR))
    if "__MACOSX" in rel or p.name.startswith("._") or p.name == ".DS_Store":
        rows.append({"path": rel, "ext": p.suffix.lower(), "size_kb": round(p.stat().st_size / 1024, 1), "md5": md5(p),
                     "role": "os-metadata (ignored)", "split": "unknown", "n_cols": None, "columns": None})
        continue
    rec = {"path": rel, "ext": p.suffix.lower(), "size_kb": round(p.stat().st_size / 1024, 1),
           "md5": md5(p), "role": "non-csv", "split": infer_split(rel), "n_cols": None, "columns": None}
    if p.suffix.lower() == ".csv":
        try:
            head = pd.read_csv(p, nrows=3)
            cols = [str(c).strip().upper() for c in head.columns]
            rec["n_cols"], rec["columns"] = len(cols), "|".join(cols)
            # A raw label file has ONLY id/stage/target columns; tables with extra columns are someone's processed features.
            if {"WAFER_ID", "AVG_REMOVAL_RATE"} <= set(cols) and set(cols) <= {"WAFER_ID", "STAGE", "AVG_REMOVAL_RATE"}:
                rec["role"] = "label"
            elif {"WAFER_ID", "TIMESTAMP"} <= set(cols):
                rec["role"] = "sensor"
            elif "AVG_REMOVAL_RATE" in cols:
                rec["role"] = "other (processed table with target)"
            else:
                rec["role"] = "other"
            if rec["role"] in ("label", "sensor") and rec["split"] == "unknown":
                rec["role"] = f"other ({rec['role']}-like, split unknown)"
        except Exception as e:
            rec["role"] = f"unreadable ({type(e).__name__})"
    rows.append(rec)

manifest = pd.DataFrame(rows)
manifest["duplicate_of_earlier_file"] = manifest.duplicated("md5", keep="first")
print(f"{len(manifest):,} files found")
display(manifest.groupby(["role", "split"]).agg(files=("path", "size"), unique_files=("duplicate_of_earlier_file", lambda s: (~s).sum()),
                                                  total_mb=("size_kb", lambda s: round(s.sum() / 1024, 1))))
display(manifest.loc[manifest.role.str.startswith(("label", "other")), ["path", "role", "split", "size_kb", "n_cols", "duplicate_of_earlier_file"]])
manifest.to_csv(OUT_DIR / "file_manifest.csv", index=False)

In [ ]:
n_dup_files = int((manifest.duplicate_of_earlier_file & manifest.role.isin(["sensor", "label"])).sum())
if n_dup_files:
    log_step("3 Raw data", "Skipped byte-identical duplicate data files", "Same file stored in more than one folder of the repo", n_dup_files, "MD5 checksum")

use_files = manifest[manifest.role.isin(["sensor", "label"]) & ~manifest.duplicate_of_earlier_file].copy()
if (use_files.role == "sensor").sum() == 0:
    raise RuntimeError("No raw sensor CSVs found – inspect `manifest` above; the data may be inside an archive format "
                       "that was not auto-extracted (e.g. .rar). Extract it into " + str(EXTRACT_DIR) + " and rerun.")

# Same file *name* with different content in two folders → possibly different versions. Report, and prefer the first.
use_files["fname"] = use_files.path.map(lambda s: Path(s).name.lower())
name_conflicts = use_files[use_files.duplicated("fname", keep=False)].sort_values("fname")
if len(name_conflicts):
    print("WARNING – same file name, different content (different versions?):")
    display(name_conflicts[["path", "md5", "size_kb"]])
    use_files = use_files.drop_duplicates("fname", keep="first")
    log_step("3 Raw data", "Kept first copy of same-named files with different content", "Avoid double-loading versions; review list above",
             len(name_conflicts) - name_conflicts.fname.nunique(), "file-name match")

print(use_files.groupby(["role", "split"]).size().rename("files used"))

### 3.2 Load the raw tables
Every sensor CSV is read as-is, column names are normalised (strip + upper-case) and two provenance columns are added: `SPLIT` and `SOURCE_FILE`. `WAFER_ID` is read **as text** so long IDs cannot be rounded or lose leading zeros. A frozen copy (`RAW_SENSOR`, `RAW_LABELS`) is kept for before/after comparisons.

In [ ]:
def read_table(rel_path, split):
    p = EXTRACT_DIR / rel_path
    head = pd.read_csv(p, nrows=0)
    id_cols = {c: str for c in head.columns if str(c).strip().upper() in ("WAFER_ID", "MACHINE_ID", "STAGE")}
    df = pd.read_csv(p, dtype=id_cols, low_memory=False)
    df.columns = [str(c).strip().upper() for c in df.columns]
    df["SPLIT"] = split
    df["SOURCE_FILE"] = Path(rel_path).name
    return df

sensor_parts, label_parts = [], []
for r in use_files.itertuples():
    (sensor_parts if r.role == "sensor" else label_parts).append(read_table(r.path, r.split))

RAW_SENSOR = pd.concat(sensor_parts, ignore_index=True)
RAW_LABELS = pd.concat(label_parts, ignore_index=True) if label_parts else pd.DataFrame(columns=["WAFER_ID", "STAGE", TARGET, "SPLIT"])
RAW_SENSOR["ROW_ORDER"] = np.arange(len(RAW_SENSOR))     # preserves original file order

print(f"Sensor rows: {len(RAW_SENSOR):,} from {len(sensor_parts)} files | Label rows: {len(RAW_LABELS):,} from {len(label_parts)} files")
print(f"Sensor memory: {RAW_SENSOR.memory_usage(deep=True).sum()/1e6:,.0f} MB")
display(RAW_SENSOR.head())
display(RAW_LABELS.head())

sensor = RAW_SENSOR.copy()     # working copies
labels = RAW_LABELS.copy()

## 4. Data dictionary & schema validation
Meanings come from the challenge documentation where available. **Units are not published** – they are recorded as "anonymised". `Role` decides how each column is treated: IDs are never used as measurements, cumulative counters are treated differently from instantaneous process signals.

In [ ]:
DICTIONARY = {
 "MACHINE_ID":                 ("id",        "Identifier of the polishing machine", "-", "string"),
 "MACHINE_DATA":               ("category",  "Machine-related code (undocumented; checked against CHAMBER in §13)", "-", "numeric code"),
 "TIMESTAMP":                  ("time",      "Time of the sensor sample", "seconds (anonymised origin)", "float"),
 "WAFER_ID":                   ("id",        "Identifier of the wafer", "-", "string"),
 "STAGE":                      ("category",  "Polishing stage (A or B)", "-", "{A, B}"),
 "CHAMBER":                    ("category",  "Chamber the wafer is in at that sample – a run passes through 3 chambers (1→2→3 or 4→5→6)", "-", "1–6"),
 "USAGE_OF_BACKING_FILM":      ("counter",   "Cumulative usage of the backing film (consumable wear)", "anonymised", ">= 0, non-decreasing until replacement"),
 "USAGE_OF_DRESSER":           ("counter",   "Cumulative usage of the pad dresser", "anonymised", ">= 0"),
 "USAGE_OF_POLISHING_TABLE":   ("counter",   "Cumulative usage of the polishing table / pad", "anonymised", ">= 0"),
 "USAGE_OF_DRESSER_TABLE":     ("counter",   "Cumulative usage of the dresser table", "anonymised", ">= 0"),
 "USAGE_OF_MEMBRANE":          ("counter",   "Cumulative usage of the head membrane", "anonymised", ">= 0"),
 "USAGE_OF_PRESSURIZED_SHEET": ("counter",   "Cumulative usage of the pressurised sheet", "anonymised", ">= 0"),
 "PRESSURIZED_CHAMBER_PRESSURE": ("process", "Pressure in the pressurised chamber", "anonymised", ">= 0"),
 "MAIN_OUTER_AIR_BAG_PRESSURE":  ("process", "Main outer air-bag pressure (head)", "anonymised", ">= 0"),
 "CENTER_AIR_BAG_PRESSURE":    ("process",   "Centre air-bag pressure (head)", "anonymised", ">= 0"),
 "RETAINER_RING_PRESSURE":     ("process",   "Retainer-ring pressure", "anonymised", ">= 0"),
 "RIPPLE_AIR_BAG_PRESSURE":    ("process",   "Ripple air-bag pressure", "anonymised", ">= 0"),
 "EDGE_AIR_BAG_PRESSURE":      ("process",   "Edge air-bag pressure", "anonymised", ">= 0"),
 "SLURRY_FLOW_LINE_A":         ("process",   "Slurry flow, line A", "anonymised", ">= 0"),
 "SLURRY_FLOW_LINE_B":         ("process",   "Slurry flow, line B", "anonymised", ">= 0"),
 "SLURRY_FLOW_LINE_C":         ("process",   "Slurry flow, line C", "anonymised", ">= 0"),
 "WAFER_ROTATION":             ("process",   "Wafer rotation speed", "anonymised", ">= 0"),
 "STAGE_ROTATION":             ("process",   "Stage (platen) rotation speed", "anonymised", ">= 0"),
 "HEAD_ROTATION":              ("process",   "Polishing-head rotation speed", "anonymised", ">= 0"),
 "DRESSING_WATER_STATUS":      ("binary",    "Dressing-water on/off status", "-", "{0, 1} expected"),
 TARGET:                       ("target",    "Average material removal rate of the run (label file)", "anonymised", "> 0"),
 "SPLIT":                      ("provenance","Organiser split the row came from (added)", "-", "{train, validation, test}"),
 "SOURCE_FILE":                ("provenance","Raw file the row came from (added)", "-", "file name"),
 "ROW_ORDER":                  ("provenance","Original row order across files (added)", "-", "int"),
}

present = set(sensor.columns) | set(labels.columns)
dd = pd.DataFrame([{"column": c, "role": r, "meaning": m, "units": u, "expected_values": e, "present": c in present}
                   for c, (r, m, u, e) in DICTIONARY.items()])
unexpected = sorted(present - set(DICTIONARY))
for c in unexpected:
    dd.loc[len(dd)] = {"column": c, "role": "unknown – review", "meaning": "Not in expected schema", "units": "?", "expected_values": "?", "present": True}
display(dd)

missing_expected = dd[(~dd.present) & (~dd.role.isin(["provenance"]))].column.tolist()
print("Expected columns missing:", missing_expected or "none")
print("Unexpected columns present:", unexpected or "none")
print("Duplicate column names – sensor:", sensor.columns[sensor.columns.duplicated()].tolist() or "none",
      "| labels:", labels.columns[labels.columns.duplicated()].tolist() or "none")
for col in ["WAFER_ID", "STAGE", "TIMESTAMP"]:
    assert col in sensor.columns, f"Required sensor column {col} missing"
assert {"WAFER_ID", TARGET} <= set(labels.columns), "Label table needs WAFER_ID and AVG_REMOVAL_RATE"

# Schema consistency across files: do all sensor files share the same columns?
schema_by_file = manifest[manifest.role == "sensor"].drop_duplicates("md5")["columns"].value_counts()
print(f"Distinct sensor column layouts across files: {len(schema_by_file)}")

COUNTERS = [c for c in dd[dd.role == "counter"].column if c in sensor.columns]
PROCESS  = [c for c in dd[dd.role == "process"].column if c in sensor.columns]
BINARY   = [c for c in dd[dd.role == "binary"].column if c in sensor.columns]
CATEG    = [c for c in ["MACHINE_ID", "MACHINE_DATA", "CHAMBER", "STAGE"] if c in sensor.columns]
NUMERIC_SIGNALS = COUNTERS + PROCESS + BINARY
print("Counters:", COUNTERS, "\nProcess signals:", PROCESS, "\nBinary:", BINARY, "\nCategorical:", CATEG)

## 5. Initial data audit (baseline profile – before any change)
This profile is saved and compared with the cleaned data in Section 23.

In [ ]:
def profile(df, name):
    out = []
    for c in df.columns:
        s = df[c]
        num = pd.to_numeric(s, errors="coerce") if s.dtype == object or str(s.dtype).startswith("str") else s
        is_num = pd.api.types.is_numeric_dtype(num) and num.notna().sum() > 0 and pd.api.types.is_numeric_dtype(s)
        out.append({"table": name, "column": c, "dtype": str(s.dtype), "n": len(s),
                    "n_missing": int(s.isna().sum()), "pct_missing": round(100 * s.isna().mean(), 3),
                    "n_unique": int(s.nunique(dropna=True)),
                    "min": num.min() if is_num else None, "p50": num.median() if is_num else None,
                    "max": num.max() if is_num else None, "mean": num.mean() if is_num else None,
                    "std": num.std() if is_num else None,
                    "top_value": s.mode(dropna=True).iloc[0] if s.notna().any() else None,
                    "sample_values": ", ".join(map(str, s.dropna().unique()[:4]))})
    return pd.DataFrame(out)

PROFILE_BEFORE = pd.concat([profile(sensor, "sensor"), profile(labels, "labels")], ignore_index=True)
display(PROFILE_BEFORE)
PROFILE_BEFORE.to_csv(OUT_DIR / "quality_profile_before.csv", index=False)

In [ ]:
print("Rows per split (sensor):"); display(sensor.groupby("SPLIT").agg(rows=("WAFER_ID", "size"), wafers=("WAFER_ID", "nunique"), files=("SOURCE_FILE", "nunique")))
print("Rows per split (labels):"); display(labels.groupby("SPLIT").agg(rows=("WAFER_ID", "size"), wafers=("WAFER_ID", "nunique")))

flags = PROFILE_BEFORE.assign(
    all_missing=PROFILE_BEFORE.pct_missing == 100,
    constant=PROFILE_BEFORE.n_unique <= 1,
    id_like_high_card=(PROFILE_BEFORE.n_unique > 0.9 * PROFILE_BEFORE.n) & (PROFILE_BEFORE.n > 100))
susp = flags[flags.all_missing | flags.constant | flags.id_like_high_card][["table", "column", "n_unique", "pct_missing", "all_missing", "constant", "id_like_high_card"]]
print("Suspicious columns (empty / constant / near-unique):"); display(susp)

runs_raw = sensor.groupby(["SPLIT", "WAFER_ID", "STAGE"], dropna=False).size().rename("n_samples")
print("Samples per run (raw):"); display(runs_raw.describe().to_frame().T)
print("Example records of the shortest and longest runs:")
display(runs_raw.sort_values().iloc[[0, -1]])

## 6. Data types, formats, units & representations
Rules applied:
* IDs (`WAFER_ID`, `MACHINE_ID`) stay **strings**, stripped of whitespace; a trailing `.0` produced by float export is removed so the same wafer matches across tables.
* `STAGE` → stripped, upper-case.
* All signal columns and `TIMESTAMP` → numeric; any value that fails to parse is **counted, shown, and set to missing** (never silently dropped).
* Units cannot be harmonised because they are not published; we check that each column has a single consistent scale per split instead (Section 14).

In [ ]:
def clean_id(s):
    s = s.astype("string").str.strip()
    return s.str.replace(r"\.0+$", "", regex=True)

for df_, name in ((sensor, "sensor"), (labels, "labels")):
    for c in ("WAFER_ID", "MACHINE_ID"):
        if c in df_.columns:
            before = df_[c].astype("string")
            df_[c] = clean_id(df_[c])
            n_changed = int((before.fillna("<NA>") != df_[c].fillna("<NA>")).sum())
            if n_changed:
                log_step("6 Types", f"Normalised {name}.{c} text (strip / remove trailing .0)", "Consistent join keys", n_changed, "string ops")
    if "STAGE" in df_.columns:
        before = df_["STAGE"].astype("string")
        df_["STAGE"] = df_["STAGE"].astype("string").str.strip().str.upper()
        n_changed = int((before.fillna("<NA>") != df_["STAGE"].fillna("<NA>")).sum())
        if n_changed:
            log_step("6 Types", f"Standardised {name}.STAGE (strip, upper-case)", "Consistent categories", n_changed, "string ops")

if "STAGE" not in labels.columns:
    print("Label table has no STAGE column → labels are per wafer; joining on WAFER_ID only.")

parse_report = []
for c in ["TIMESTAMP", "MACHINE_DATA", "CHAMBER"] + NUMERIC_SIGNALS:
    if c not in sensor.columns:
        continue
    raw = sensor[c]
    conv = pd.to_numeric(raw, errors="coerce")
    failed = raw.notna() & conv.isna()
    parse_report.append({"column": c, "orig_dtype": str(raw.dtype), "parse_failures": int(failed.sum()),
                         "examples": ", ".join(map(str, raw[failed].unique()[:5]))})
    if failed.any():
        log_step("6 Types", f"Set {int(failed.sum())} unparseable values in {c} to missing", "Non-numeric text in numeric column", failed.sum(), "pd.to_numeric(errors='coerce')")
    sensor[c] = conv

labels[TARGET] = pd.to_numeric(labels[TARGET], errors="coerce")
display(pd.DataFrame(parse_report))

# Integer-valued codes → nullable integers
for c in ["CHAMBER", "MACHINE_DATA"] + BINARY:
    if c in sensor.columns and sensor[c].dropna().mod(1).eq(0).all():
        sensor[c] = sensor[c].astype("Int64")
print(sensor.dtypes.to_string())

## 7. Validate values and relationships between fields
Each rule is evaluated and its violation count reported. Impossible values (e.g. negative pressure) are distinguished from unusual-but-possible ones (handled in Section 11).

In [ ]:
rules = []
def rule(name, mask, table="sensor", severity="error", note=""):
    n = int(mask.sum())
    rules.append({"table": table, "rule": name, "violations": n, "pct": round(100 * n / max(len(mask), 1), 4), "severity": severity, "note": note})
    return mask

nonneg_cols = [c for c in COUNTERS + PROCESS if c in sensor.columns]
neg_masks = {c: rule(f"{c} >= 0", sensor[c] < 0) for c in nonneg_cols}
rule("STAGE in {A,B}", ~sensor["STAGE"].isin(["A", "B"]) & sensor["STAGE"].notna())
for c in BINARY:
    rule(f"{c} in {{0,1}}", ~sensor[c].isin([0, 1]) & sensor[c].notna(), severity="warning",
         note="observed values: " + ", ".join(map(str, sorted(sensor[c].dropna().unique())[:8])))
rule("TIMESTAMP not missing", sensor["TIMESTAMP"].isna())

# Cross-field: time must not go backwards inside a run (in original file order)
sensor = sensor.sort_values("ROW_ORDER")
dt_in_file = sensor.groupby(RUN_KEY, dropna=False)["TIMESTAMP"].diff()
rule("TIMESTAMP non-decreasing within run (file order)", dt_in_file < 0, severity="warning", note="fixed by sorting")

# Cross-field: a run moves through several chambers (wafer transfer), so CHAMBER is NOT constant inside a run.
# The meaningful run-level property is the chamber *group*: all chambers in {1,2,3} or all in {4,5,6}.
if "CHAMBER" in sensor.columns:
    ch_sets = sensor.groupby(RUN_KEY, dropna=False)["CHAMBER"].agg(lambda s: tuple(sorted(s.dropna().unique())))
    print("Chamber sequences per run (top 8):"); display(ch_sets.value_counts().head(8).to_frame("runs"))
    def ch_group(t):
        if len(t) and all(x <= 3 for x in t): return "123"
        if len(t) and all(x >= 4 for x in t): return "456"
        return "mixed"
    RUN_CH_GROUP = ch_sets.map(ch_group).rename("CHAMBER_GROUP")
    rules.append({"table": "sensor-run", "rule": "run stays within one chamber group (1-3 or 4-6)", "violations": int((RUN_CH_GROUP == "mixed").sum()),
                  "pct": round(100 * (RUN_CH_GROUP == "mixed").mean(), 4), "severity": "warning", "note": "chamber changes inside a run are expected"})
    sensor = sensor.join(RUN_CH_GROUP, on=RUN_KEY)
    # Operating mode = stage × chamber group. Defined from sensor data only (never from the target).
    sensor["MODE"] = sensor["STAGE"].astype(str) + "-" + sensor["CHAMBER_GROUP"].astype(str)
else:
    sensor["MODE"] = sensor["STAGE"].astype(str)
for c in ("MACHINE_ID",):
    if c in sensor.columns:
        k = sensor.groupby(RUN_KEY, dropna=False)[c].nunique()
        rules.append({"table": "sensor-run", "rule": f"single {c} per run", "violations": int((k > 1).sum()),
                      "pct": round(100 * (k > 1).mean(), 4), "severity": "warning", "note": ""})

# Cross-field: counters should not decrease inside a run (they only reset between runs when a part is replaced)
for c in COUNTERS:
    d = sensor.groupby(RUN_KEY, dropna=False)[c].diff()
    rule(f"{c} non-decreasing within run", d < 0, severity="warning", note="decrease inside a run is physically odd")

# Labels
rule(f"{TARGET} > 0", labels[TARGET] <= 0, table="labels")
rule(f"{TARGET} not missing", labels[TARGET].isna(), table="labels", note="labels missing ⇒ run unusable for supervised learning")

RULES = pd.DataFrame(rules)
display(RULES)

print("Runs per operating mode (stage × chamber group) and split:")
display(sensor.drop_duplicates(RUN_KEY).pivot_table(index="MODE", columns="SPLIT", values="WAFER_ID", aggfunc="count", fill_value=0))

In [ ]:
# Treatments
n_neg = int(sum(m.sum() for m in neg_masks.values()))
for c, m in neg_masks.items():
    if m.any():
        sensor.loc[m, c] = np.nan
if n_neg:
    log_step("7 Validation", "Negative counter/process values set to missing", "Physically impossible for usage, pressure, flow, rotation", n_neg, "rule >= 0")

n_back = int((dt_in_file < 0).sum())
sensor = sensor.sort_values(RUN_KEY + ["TIMESTAMP", "ROW_ORDER"]).reset_index(drop=True)
if n_back:
    log_step("7 Validation", "Re-sorted samples by TIMESTAMP within each run", "Timestamps out of order in file", n_back, "stable sort")

bad_lab = labels[TARGET] <= 0
if bad_lab.any():
    labels.loc[bad_lab, TARGET] = np.nan
    log_step("7 Validation", f"Non-positive {TARGET} set to missing", "Removal rate must be positive", bad_lab.sum())
RULES.to_csv(OUT_DIR / "validation_rules.csv", index=False)

## 8. Missing data
1. Search for **all** representations of missingness: real NaN, text tokens (handled in §6), and numeric **sentinels** (−999, 9999 …).
2. Profile missingness by column, by split, by run, and check whether columns are missing **together**.
3. Check whether **zero** is plausible or a disguised "not recorded" (e.g. a whole run of zero pressure).
4. Decide treatment per column and explain it.

In [ ]:
SENTINELS = [-9999, -999, -99, 9999, 99999, 999999]
sent = []
for c in NUMERIC_SIGNALS:
    s = sensor[c]
    for v in SENTINELS:
        n = int((s == v).sum())
        if n:
            sent.append({"column": c, "sentinel": v, "count": n})
SENT = pd.DataFrame(sent)
print("Sentinel-looking values:"); display(SENT if len(SENT) else "none found")
for r in sent:
    # A sentinel that is also far outside the column's normal range is treated as missing
    s = sensor[r["column"]]
    q1, q99 = s.quantile([0.01, 0.99])
    if r["sentinel"] < q1 or r["sentinel"] > q99:
        sensor.loc[s == r["sentinel"], r["column"]] = np.nan
        log_step("8 Missing", f"Sentinel {r['sentinel']} in {r['column']} → missing", "Value far outside normal range, typical missing code", r["count"])

In [ ]:
miss_col = sensor[NUMERIC_SIGNALS + CATEG + ["TIMESTAMP"]].isna().groupby(sensor["SPLIT"]).mean().T * 100
print("% missing by column and split:"); display(miss_col[miss_col.sum(axis=1) > 0] if (miss_col.values > 0).any() else "no missing values in sensor table")

if (miss_col.values > 0).any():
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(miss_col))))
    sns.heatmap(miss_col, cmap="Blues", annot=True, fmt=".2f", cbar_kws={"label": "% missing"}, ax=ax)
    ax.set_title("Missing values by column and split (%)")
    save_fig("08_missing_by_split")
    m = sensor[NUMERIC_SIGNALS].isna()
    co = m.loc[:, m.any()].astype(int).corr()
    if co.shape[0] > 1:
        print("Do columns go missing together? (correlation of missingness indicators)"); display(co.round(2))

run_miss = sensor.groupby(RUN_KEY, dropna=False)[NUMERIC_SIGNALS].apply(lambda g: g.isna().mean().mean() * 100).rename("pct_cells_missing")
print("Runs with any missing cell:", int((run_miss > 0).sum()), "of", len(run_miss))

# Zero runs: a whole run where a process signal is exactly zero is suspicious ("not recorded" disguised as 0)
zero_runs = (sensor[PROCESS].fillna(0) == 0).groupby([sensor[k] for k in RUN_KEY], dropna=False).all()
zr = zero_runs.sum().sort_values(ascending=False)
print("Runs in which a process signal is zero for the entire run (possible 'not recorded'):"); display(zr[zr > 0].to_frame("runs") if zr.sum() else "none")

PRESSURES = [c for c in PROCESS if "PRESSURE" in c]
ZERO_PRESSURE_RUN = zero_runs[PRESSURES].all(axis=1).rename("ZERO_PRESSURE_RUN") if PRESSURES else pd.Series(dtype=bool)
if ZERO_PRESSURE_RUN.any():
    zp = ZERO_PRESSURE_RUN.reset_index().merge(sensor[RUN_KEY + ["MODE"]].drop_duplicates(), on=RUN_KEY)
    print("Runs with ALL pressures = 0 for the entire run, by mode and split:")
    display(zp[zp.ZERO_PRESSURE_RUN].groupby(["MODE", "SPLIT"]).size().unstack(fill_value=0))
    # Test the 'not recorded' hypothesis: if the polishing really had zero pressure, removal rate would collapse.
    lk = [c for c in ["SPLIT", "WAFER_ID", "STAGE"] if c in labels.columns]
    zt = zp.merge(labels[lk + [TARGET]], on=lk, how="inner")
    zt = zt[zt.MODE.isin(zp.loc[zp.ZERO_PRESSURE_RUN, "MODE"].unique())]
    cmp_zero = zt.groupby(["MODE", "ZERO_PRESSURE_RUN"])[TARGET].agg(["size", "median"])
    print("Removal rate of zero-pressure runs vs other runs in the same mode:"); display(cmp_zero)
    print("→ If medians are similar, the wafers WERE polished normally: the zeros mean 'pressure not recorded', not 'no pressure'.")

In [ ]:
# Labels: which runs have sensors but no label, and vice versa (evaluated properly in §10)
print("Missing target values:", int(labels[TARGET].isna().sum()))

# Treatment decision (documented):
#  * Sensor time series: missing samples are filled *within the same run only* by linear interpolation in time
#    (signals are smooth, sampled every few seconds). Gaps longer than MAX_INTERP samples are left missing.
#  * Nothing is ever filled across runs or with zero.
#  * Runs whose target is missing are kept for EDA of sensors but excluded from supervised analysis — target is never imputed.
MAX_INTERP = 3
n_before = int(sensor[NUMERIC_SIGNALS].isna().sum().sum())
if n_before:
    sensor["MISSING_ANY_SIGNAL"] = sensor[NUMERIC_SIGNALS].isna().any(axis=1)   # missingness indicator kept
    sensor[NUMERIC_SIGNALS] = (sensor.groupby(RUN_KEY, dropna=False)[NUMERIC_SIGNALS]
                               .transform(lambda s: s.interpolate(limit=MAX_INTERP, limit_area="inside")))
    n_after = int(sensor[NUMERIC_SIGNALS].isna().sum().sum())
    log_step("8 Missing", f"Interpolated short gaps (≤{MAX_INTERP} samples) inside runs", "Smooth sensor signals; gaps are short",
             n_before - n_after, "linear interpolation within run, inside only")
    print(f"Missing cells: {n_before:,} → {n_after:,} (remaining are left as NaN and handled by run-level aggregation)")
else:
    sensor["MISSING_ANY_SIGNAL"] = False
    print("No missing sensor cells – no imputation needed.")

## 9. Duplicates – at the correct level
* **Exact duplicate rows** (all columns identical apart from provenance).
* **Key duplicates**: the same `(SPLIT, WAFER_ID, STAGE, TIMESTAMP)` appearing twice with *different* values → conflict.
* **Label duplicates**: the same run labelled twice; conflicting labels.
* **Cross-split**: a wafer ID present in more than one split (leakage risk – see §12).
Repeated `WAFER_ID` across rows is **expected** (one run = many samples) and is not a duplicate.

In [ ]:
content_cols = [c for c in sensor.columns if c not in ("SOURCE_FILE", "ROW_ORDER", "MISSING_ANY_SIGNAL")]
exact = sensor.duplicated(content_cols, keep="first")
print("Exact duplicate sensor rows:", int(exact.sum()))
if exact.any():
    display(sensor[sensor.duplicated(content_cols, keep=False)].head(6))
    sensor = sensor[~exact].reset_index(drop=True)
    log_step("9 Duplicates", "Removed exact duplicate sensor rows", "Same sample recorded twice", exact.sum(), "keep first")

key = RUN_KEY + ["TIMESTAMP"]
kdup = sensor.duplicated(key, keep=False)
print("Rows sharing a (run, TIMESTAMP) key but differing in values:", int(kdup.sum()))
if kdup.any():
    display(sensor[kdup].sort_values(key).head(6))
    agg = {c: "mean" for c in NUMERIC_SIGNALS}
    n_before = len(sensor)
    first_cols = [c for c in sensor.columns if c not in NUMERIC_SIGNALS + key]
    sensor = (sensor.groupby(key, dropna=False, as_index=False)
                    .agg({**agg, **{c: "first" for c in first_cols}}))
    log_step("9 Duplicates", "Averaged conflicting samples with identical run+timestamp", "Two readings of the same instant; mean is the least-biased merge",
             n_before - len(sensor), "groupby mean")

lab_key = [c for c in ["SPLIT", "WAFER_ID", "STAGE"] if c in labels.columns]
lex = labels.duplicated(lab_key + [TARGET])
if lex.any():
    labels = labels[~lex].reset_index(drop=True)
    log_step("9 Duplicates", "Removed exact duplicate label rows", "Same label listed twice", lex.sum())
lconf = labels.duplicated(lab_key, keep=False)
print("Runs with conflicting labels:", int(lconf.sum()))
if lconf.any():
    display(labels[lconf].sort_values(lab_key))
    labels = labels[~lconf].reset_index(drop=True)
    log_step("9 Duplicates", "Dropped runs with conflicting labels", "Cannot tell which removal-rate value is correct", lconf.sum())

w_split = sensor.groupby("WAFER_ID")["SPLIT"].nunique()
cross = w_split[w_split > 1]
print("Wafer IDs appearing in more than one split:", len(cross))
if len(cross):
    display(sensor[sensor.WAFER_ID.isin(cross.index)].groupby(["WAFER_ID", "SPLIT", "STAGE"]).size().head(10))

stages_per_wafer = sensor.groupby(["SPLIT", "WAFER_ID"])["STAGE"].nunique().value_counts()
print("Stages per wafer (count of wafers):"); display(stages_per_wafer.to_frame("wafers"))

## 10. Joins & integration: sensor runs ↔ labels
Expected relationship: **one sensor run ↔ exactly one label** (1:1 at run level; 1:many at sample level). We check key uniqueness, match rates in both directions, and – after the merge in §15 – that the row count did not multiply.

In [ ]:
sensor_runs = sensor[lab_key].drop_duplicates()
label_runs = labels[lab_key + [TARGET]]
assert not label_runs.duplicated(lab_key).any(), "Label key not unique – join would multiply rows"

chk = sensor_runs.merge(label_runs, on=lab_key, how="outer", indicator=True)
match = chk.groupby("SPLIT")["_merge"].value_counts().unstack(fill_value=0)
match["match_rate_%"] = (100 * match.get("both", 0) / match.sum(axis=1)).round(2)
display(match)
UNLABELLED_RUNS = chk[chk._merge == "left_only"][lab_key]
ORPHAN_LABELS = chk[chk._merge == "right_only"][lab_key]
print(f"Sensor runs without label: {len(UNLABELLED_RUNS):,} | labels without sensor data: {len(ORPHAN_LABELS):,}")
if len(UNLABELLED_RUNS):
    print("Unlabelled runs by split:", UNLABELLED_RUNS.SPLIT.value_counts().to_dict(),
          "→ kept for sensor EDA, excluded from target analysis (the organisers may withhold some labels).")
if len(ORPHAN_LABELS):
    log_step("10 Join", "Labels without any sensor run will be ignored", "Nothing to describe the run", len(ORPHAN_LABELS))

## 11. Outliers
**Target outliers** are detected with a robust (median/MAD) z-score *within operating mode (stage × chamber group)*, because modes run at different removal-rate levels – a value normal for one mode can be extreme for another. (A stage-only rule would flag an entire genuine high-rate mode as outliers.) We also apply the IQR rule for comparison. Rows are **flagged, not deleted**: the main analysis excludes flagged training labels and §21 checks how much that choice matters.

**Sensor outliers** are handled later at the run level (§14), because individual sensor spikes are usually transient.

In [ ]:
def robust_z(s):
    med = s.median()
    mad = (s - med).abs().median() * 1.4826
    return (s - med) / (mad if mad > 0 else s.std() if s.std() > 0 else 1)

# Outlier groups = operating mode (stage × chamber) when available: a stage can contain several genuine
# removal-rate modes, and a stage-only rule would wrongly flag a whole high-rate mode as "outliers".
if "STAGE" in labels.columns:
    run_mode = sensor.groupby(lab_key, dropna=False)["MODE"].first().rename("MODE_OF_RUN").reset_index()
    labels = labels.drop(columns=["MODE_OF_RUN", "OUTLIER_GROUP"], errors="ignore").merge(run_mode, on=lab_key, how="left")
    labels["OUTLIER_GROUP"] = labels["MODE_OF_RUN"].fillna(labels["STAGE"].astype(str))
    small = labels["OUTLIER_GROUP"].map(labels["OUTLIER_GROUP"].value_counts()) < 20      # too few runs → fall back to stage
    labels.loc[small, "OUTLIER_GROUP"] = labels.loc[small, "STAGE"].astype(str)
    grp = "OUTLIER_GROUP"
else:
    grp = None
print("Outlier detection groups:", labels[grp].value_counts().to_dict() if grp else "whole dataset")
labels["TARGET_ROBUST_Z"] = labels.groupby(grp)[TARGET].transform(robust_z) if grp else robust_z(labels[TARGET])
q1 = labels.groupby(grp)[TARGET].transform(lambda s: s.quantile(0.25)) if grp else labels[TARGET].quantile(0.25)
q3 = labels.groupby(grp)[TARGET].transform(lambda s: s.quantile(0.75)) if grp else labels[TARGET].quantile(0.75)
iqr = q3 - q1
labels["TARGET_IQR_OUT"] = (labels[TARGET] < q1 - 3 * iqr) | (labels[TARGET] > q3 + 3 * iqr)   # 3×IQR = "far out"
Z_LIMIT = 5.0   # conservative: only flag clearly impossible values; data has genuine multi-modality
labels["TARGET_OUTLIER"] = labels["TARGET_ROBUST_Z"].abs() > Z_LIMIT

print(f"Flagged by robust z>|{Z_LIMIT}|: {int(labels.TARGET_OUTLIER.sum())} | by 3×IQR: {int(labels.TARGET_IQR_OUT.sum())}")
display(labels[labels.TARGET_OUTLIER | labels.TARGET_IQR_OUT].sort_values(TARGET, ascending=False).head(20))
print("Target summary by split and stage:")
display(labels.groupby(["SPLIT"] + (["STAGE"] if "STAGE" in labels else []))[TARGET].describe(percentiles=[.01, .25, .5, .75, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=labels, x="SPLIT", y=TARGET, hue="STAGE" if "STAGE" in labels else None, ax=axes[0], linewidth=1, fliersize=3)
axes[0].set_title("Removal rate – all labels (log scale)"); axes[0].set_yscale("log")
sns.boxplot(data=labels[~labels.TARGET_OUTLIER], x="SPLIT", y=TARGET, hue="STAGE" if "STAGE" in labels else None, ax=axes[1], linewidth=1, fliersize=3)
axes[1].set_title("Removal rate – after excluding flagged outliers")
save_fig("11_target_outliers")

n_out = int(labels.TARGET_OUTLIER.sum())
if n_out:
    log_step("11 Outliers", f"Flagged {n_out} labels with |robust z| > {Z_LIMIT} within operating mode",
             "Values far outside their operating mode's distribution ⇒ likely recording errors; excluded from main training/EDA, kept in file",
             n_out, "median/MAD z-score per mode (fallback: stage)")

## 12. Leakage prevention & split strategy (decided *before* any fitted transformation)

| Decision | Choice & reason |
|---|---|
| Splits | Use the organiser **train / validation / test** splits as given. Train = fit; validation = model selection; test = touched **once** at the end. |
| Grouping | Many samples per run ⇒ aggregate to run level first; CV inside train uses **GroupKFold by `WAFER_ID`** so the same wafer (stage A + B) never sits in both folds. |
| Fitted transforms | Imputer, scaler, encoders live inside an sklearn `Pipeline` fitted on training folds only. |
| Outlier rule | Robust-z thresholds above only *flag*; for modelling the threshold is **re-derived from training labels only**. |
| Target leakage | `AVG_REMOVAL_RATE` and anything derived from it (z-score, outlier flags) are never features. |
| Future information | Run features use only samples of that run. Neighbouring-wafer features (known to help on this dataset) would use other wafers' *labels* – allowed only if those wafers were measured **earlier** than the run being predicted; not used here. |
| Cross-split duplicates | Checked in §9 (wafer IDs present in more than one split are listed). |
| EDA on full data | Descriptive statistics over all splits describe the dataset; they are **not** used to tune anything. |

In [ ]:
train_labels = labels[labels.SPLIT == "train"]
if grp and len(train_labels):
    tr_stats = train_labels.groupby(grp)[TARGET].agg(med="median", mad=lambda s: (s - s.median()).abs().median() * 1.4826)
    lb = labels.join(tr_stats, on=grp)
    labels["OUTLIER_TRAIN_RULE"] = ((lb[TARGET] - lb["med"]).abs() / lb["mad"].replace(0, np.nan)) > Z_LIMIT
else:
    labels["OUTLIER_TRAIN_RULE"] = labels["TARGET_OUTLIER"]
print("Outliers using thresholds fitted on TRAIN only, by split:"); display(labels.groupby("SPLIT")["OUTLIER_TRAIN_RULE"].sum())
print("Same wafer in several splits (should be 0):", len(cross))

## 13. Categorical variables
`MACHINE_ID`, `MACHINE_DATA`, `CHAMBER`, `STAGE` are categories even when stored as numbers. We inspect frequencies per split, rare levels, and levels that appear only in validation/test (**unseen categories** a model must handle).

In [ ]:
for c in CATEG:
    tab = sensor.drop_duplicates(RUN_KEY).groupby("SPLIT")[c].value_counts(dropna=False).unstack(0, fill_value=0)
    tab["share_%"] = (100 * tab.sum(axis=1) / tab.values.sum()).round(2)
    print(f"\n{c}: {sensor[c].nunique()} levels (counted in runs)"); display(tab.sort_values("share_%", ascending=False).head(15))
    tr_levels = set(sensor.loc[sensor.SPLIT == "train", c].dropna().unique())
    for sp in ("validation", "test"):
        unseen = set(sensor.loc[sensor.SPLIT == sp, c].dropna().unique()) - tr_levels
        if unseen:
            print(f"  ⚠ levels in {sp} not seen in train: {sorted(map(str, unseen))[:10]}")

# Is MACHINE_DATA just a copy of the starting chamber?
if {"MACHINE_DATA", "CHAMBER"} <= set(sensor.columns):
    g0 = sensor.groupby(RUN_KEY, dropna=False)
    same = (g0["MACHINE_DATA"].first().astype(str) == g0["CHAMBER"].first().astype(str)).mean()
    print(f"\nMACHINE_DATA equals the run's first CHAMBER in {100*same:.1f}% of runs")
    MACHINE_DATA_REDUNDANT = same > 0.99
else:
    MACHINE_DATA_REDUNDANT = False

# Segment variable used for EDA & modelling: MODE = stage × chamber group (defined in §7 from sensor data, not from the target).
# CHAMBER itself changes inside a run, so it is not a run-level category; MACHINE_ID is constant; MACHINE_DATA may be redundant.
CATEG_FEATS = [c for c in ["STAGE", "MODE", "MACHINE_ID"] + ([] if MACHINE_DATA_REDUNDANT else ["MACHINE_DATA"]) if c in sensor.columns]
if MACHINE_DATA_REDUNDANT:
    log_step("13 Categorical", "MACHINE_DATA not used as a feature", "Identical to the run's starting chamber, already captured by MODE", None)
print("\nEncoding plan: one-hot (nominal, no order) with handle_unknown='ignore'; constant / single-level columns dropped.")

## 14. Numerical variables
Distribution shape, skew, scale and constant columns of the raw signals. (Scaling, if any, happens inside the modelling pipeline; tree models need none. Reporting stays in original units.)

In [ ]:
num_desc = sensor[NUMERIC_SIGNALS].describe(percentiles=[.01, .5, .99]).T
num_desc["skew"] = sensor[NUMERIC_SIGNALS].skew()
num_desc["n_unique"] = sensor[NUMERIC_SIGNALS].nunique()
num_desc["zero_%"] = (sensor[NUMERIC_SIGNALS] == 0).mean() * 100
display(num_desc)
CONSTANT_SIGNALS = num_desc.index[num_desc.n_unique <= 1].tolist()
print("Constant signals (no information):", CONSTANT_SIGNALS or "none")

cols = [c for c in NUMERIC_SIGNALS if c not in CONSTANT_SIGNALS]
ncol = 4; nrow = int(np.ceil(len(cols) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.8 * nrow))
samp = sensor.sample(min(len(sensor), 200_000), random_state=RANDOM_STATE)
for ax, c in zip(axes.ravel(), cols):
    sns.histplot(samp[c].dropna(), bins=60, ax=ax, color=PALETTE[0], edgecolor="white", linewidth=0.3)
    ax.set_title(c, fontsize=9); ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes.ravel()[len(cols):]:
    ax.axis("off")
fig.suptitle("Raw sensor signal distributions (sample-level, up to 200k rows)", y=1.0)
save_fig("14_signal_histograms")
print("Many process signals are multi-modal: the machine switches between set-points (idle / ramp / polish) within a run, "
      "so a single mean per run can hide the polishing phase – motivating per-run percentiles and std in §15.")

# Same scale in every split? (sign of unit or calibration change)
scale = sensor.groupby("SPLIT")[cols].median().T
print("Median of each signal by split (large ratios would hint at unit/calibration differences):"); display(scale)

## 17. Time-series-specific checks *(placed before feature engineering because features depend on them)*
Ordering (fixed in §7), duplicate timestamps (§9), **sampling interval**, **gaps**, **run duration**, and how counters evolve across consecutive runs (consumable replacement events).

In [ ]:
sensor["DT"] = sensor.groupby(RUN_KEY, dropna=False)["TIMESTAMP"].diff()
ts = sensor.groupby(RUN_KEY, dropna=False).agg(n_samples=("TIMESTAMP", "size"), t_start=("TIMESTAMP", "min"), t_end=("TIMESTAMP", "max"),
                                                median_dt=("DT", "median"), max_dt=("DT", "max"))
ts["duration"] = ts.t_end - ts.t_start
# Logging slows down during some phases (intervals of ~15-17 units are routine), so a "gap" must be judged against the
# global distribution of intervals, not only the run median: gap = interval above the 99.9th percentile of all intervals.
GAP_LIMIT = max(3 * float(sensor.DT.median()), float(sensor.DT.quantile(0.999)))
ts["n_gaps"] = (sensor.DT > GAP_LIMIT).groupby([sensor[k] for k in RUN_KEY], dropna=False).sum()
ts = ts.join(sensor.groupby(RUN_KEY, dropna=False)["MODE"].first())
display(ts.describe().T)
print(f"Typical sampling interval: {ts.median_dt.median():.3g} | gap threshold: {GAP_LIMIT:.3g} | runs with ≥1 gap: {int((ts.n_gaps > 0).sum())}")

# Abnormally long runs: > 2× the median duration of their mode. These usually hold several polishing passes
# (rework, or a wafer ID reused) under one ID, while the label is one average – their features are not comparable.
mode_med = ts.groupby("MODE")["duration"].transform("median")
ts["LONG_RUN"] = ts.duration > 2 * mode_med
print(f"Abnormally long runs (> 2× mode median duration): {int(ts.LONG_RUN.sum())}")
display(ts[ts.LONG_RUN].sort_values("duration", ascending=False).head(10))
if ts.LONG_RUN.any():
    log_step("17 Time series", "Flagged abnormally long runs (LONG_RUN)", "Likely several passes under one ID; excluded from model training, kept for EDA", ts.LONG_RUN.sum(), "duration > 2× mode median")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
sns.histplot(ts[~ts.LONG_RUN].reset_index(), x="n_samples", hue="SPLIT", bins=50, ax=axes[0], palette=SPLIT_COLORS, element="step", fill=False)
axes[0].set_title("Samples per run")
sns.histplot(ts[~ts.LONG_RUN].reset_index(), x="duration", hue="SPLIT", bins=50, ax=axes[1], palette=SPLIT_COLORS, element="step", fill=False)
axes[1].set_title("Run duration (long runs excluded)")
sns.histplot(sensor.DT.dropna().clip(upper=30), bins=60, ax=axes[2], color=PALETTE[0])
axes[2].set_yscale("log"); axes[2].set_title("Sampling intervals (clipped at 30, log count)")
save_fig("17_run_timing")

too_short = ts.n_samples < max(5, ts.n_samples.quantile(0.01) * 0.5)
print("Very short runs (incomplete sequences?):", int(too_short.sum()))
SHORT_RUNS = ts.index[too_short]

# Example run: what does one polishing run look like?
ex = sensor.groupby(RUN_KEY, dropna=False).ngroup()
ex_run = sensor[ex == ex.iloc[len(ex) // 2]]
show = [c for c in ["CENTER_AIR_BAG_PRESSURE", "RETAINER_RING_PRESSURE", "HEAD_ROTATION", "SLURRY_FLOW_LINE_A"] if c in cols][:4] or cols[:4]
fig, axes = plt.subplots(len(show), 1, figsize=(11, 1.9 * len(show)), sharex=True)
for ax, c, col in zip(np.atleast_1d(axes), show, PALETTE):
    ax.plot(ex_run.TIMESTAMP - ex_run.TIMESTAMP.min(), ex_run[c], color=col, lw=1.6); ax.set_ylabel(c, fontsize=8)
np.atleast_1d(axes)[0].set_title(f"One run: wafer {ex_run.WAFER_ID.iloc[0]}, stage {ex_run.STAGE.iloc[0]} – signals vs elapsed time")
np.atleast_1d(axes)[-1].set_xlabel("elapsed time in run")
save_fig("17_example_run")

## 15. Feature engineering – one row per polishing run
Every feature uses **only samples of that run** (available at prediction time).

| Feature | Formula | Why |
|---|---|---|
| `<counter>_start` | first value of each `USAGE_OF_*` in the run | consumable age when the wafer was polished |
| `<counter>_delta` | last − first | wear accumulated during the run |
| `<signal>_mean/std/p10/p50/p90` | per process signal | level, stability, and polishing-phase set-point (p90) |
| `<signal>_active_mean` | mean over samples where head rotation > 0 (fallback: all) | isolates the actual polishing phase |
| `<binary>_frac_on` | share of samples with status = 1 | dressing activity |
| `n_samples`, `duration`, `median_dt`, `n_gaps` | timing | run length, data completeness |
| `TOTAL_SLURRY_FLOW_mean` | mean of A+B+C | total slurry supply |
| `HEAD_STAGE_ROT_RATIO` | head_rotation_mean / stage_rotation_mean (NaN if denominator 0) | relative kinematics |
| `MISSING_SIGNAL_FRAC` | share of samples with any missing signal before interpolation | data quality indicator |

In [ ]:
proc_cols = [c for c in PROCESS if c not in CONSTANT_SIGNALS]
g = sensor.groupby(RUN_KEY, dropna=False)

parts = [ts[["n_samples", "duration", "median_dt", "n_gaps", "t_start"]]]
cat_first = {c: (c, "first") for c in CATEG_FEATS + ["CHAMBER"] if c not in ("STAGE",) and c in sensor.columns}
if cat_first:
    parts.append(g.agg(**cat_first))
if COUNTERS:
    first = g[COUNTERS].first().add_suffix("_start")
    last = g[COUNTERS].last()
    parts += [first, (last - g[COUNTERS].first()).add_suffix("_delta")]
if proc_cols:
    parts.append(g[proc_cols].agg(["mean", "std"]).pipe(lambda d: d.set_axis([f"{a}_{b}" for a, b in d.columns], axis=1)))
    for q in (0.1, 0.5, 0.9):
        parts.append(g[proc_cols].quantile(q).add_suffix(f"_p{int(q*100)}"))
    if "HEAD_ROTATION" in sensor.columns:
        act = sensor[sensor.HEAD_ROTATION > 0].groupby(RUN_KEY, dropna=False)[proc_cols].mean().add_suffix("_active_mean")
        parts.append(act)
for c in BINARY:
    parts.append(g[c].apply(lambda s: (s == 1).mean()).rename(f"{c}_frac_on"))
parts.append(g["MISSING_ANY_SIGNAL"].mean().rename("MISSING_SIGNAL_FRAC"))

runs = pd.concat(parts, axis=1).reset_index()
slurry = [c for c in ["SLURRY_FLOW_LINE_A", "SLURRY_FLOW_LINE_B", "SLURRY_FLOW_LINE_C"] if f"{c}_mean" in runs]
if slurry:
    runs["TOTAL_SLURRY_FLOW_mean"] = runs[[f"{c}_mean" for c in slurry]].sum(axis=1, min_count=1)
if {"HEAD_ROTATION_mean", "STAGE_ROTATION_mean"} <= set(runs.columns):
    runs["HEAD_STAGE_ROT_RATIO"] = runs.HEAD_ROTATION_mean / runs.STAGE_ROTATION_mean.replace(0, np.nan)
for c in [x for x in runs.columns if x.endswith("_active_mean")]:
    runs[c] = runs[c].fillna(runs[c.replace("_active_mean", "_mean")])   # fallback when no active phase

runs["SHORT_RUN"] = runs.set_index(RUN_KEY).index.isin(SHORT_RUNS)
runs = runs.merge(ts["LONG_RUN"].reset_index(), on=RUN_KEY, how="left")

# Signals that are 0 for an entire run were shown in §8 to mean "not recorded" → their run aggregates become NaN
# (never treated as a real 0), and an indicator is kept. Imputation happens later inside the model pipeline.
if len(zero_runs):
    zr_ = zero_runs.reset_index()
    runs = runs.merge(ZERO_PRESSURE_RUN.reset_index(), on=RUN_KEY, how="left")
    zm = runs[RUN_KEY].merge(zr_, on=RUN_KEY, how="left")
    n_set = 0
    for sig in PROCESS:
        if sig not in zm or not ZERO_PRESSURE_RUN.any(): continue
        mask = zm[sig].fillna(False).astype(bool).values & runs["ZERO_PRESSURE_RUN"].fillna(False).astype(bool).values
        cols_sig = [c for c in runs.columns if c.startswith(sig + "_")]
        runs.loc[mask, cols_sig] = np.nan; n_set += int(mask.sum())
    for c in ["TOTAL_SLURRY_FLOW_mean", "HEAD_STAGE_ROT_RATIO"]:
        if c in runs: runs.loc[runs.ZERO_PRESSURE_RUN.fillna(False).astype(bool), c] = np.nan
    if n_set:
        log_step("15 Features", "Run aggregates of all-zero signals in zero-pressure runs set to NaN", "Zeros = not recorded (§8); a 0 would fake a real low value",
                 int(runs.ZERO_PRESSURE_RUN.sum()), "flag ZERO_PRESSURE_RUN + NaN")
print(f"Run-level table: {runs.shape[0]:,} runs × {runs.shape[1]} columns (expected runs: {len(ts):,})")
assert len(runs) == len(ts)

# ---- Join with labels (validated 1:1) ----
n_before = len(runs)
lab_cols = lab_key + [TARGET, "TARGET_OUTLIER", "OUTLIER_TRAIN_RULE", "TARGET_ROBUST_Z"]
runs = runs.merge(labels[lab_cols], on=lab_key, how="left", validate="one_to_one" if "STAGE" in lab_key else "many_to_one")
assert len(runs) == n_before, "Join multiplied rows!"
print(f"Rows before join {n_before:,} → after {len(runs):,} (no multiplication) | runs with label: {runs[TARGET].notna().sum():,}")
log_step("15 Features", "Aggregated sensor samples to one row per run and joined labels", "Target is defined per run", len(runs), "groupby run + validated 1:1 merge")

FEATURES = [c for c in runs.columns if c not in RUN_KEY + [TARGET, "TARGET_OUTLIER", "OUTLIER_TRAIN_RULE", "TARGET_ROBUST_Z", "t_start", "SHORT_RUN", "LONG_RUN", "CHAMBER"]]
for c in ("ZERO_PRESSURE_RUN",):
    if c in runs: runs[c] = runs[c].fillna(False).astype(int)
NUM_FEATURES = [c for c in FEATURES if pd.api.types.is_numeric_dtype(runs[c]) and c not in CATEG_FEATS]
CAT_FEATURES = [c for c in FEATURES if c in CATEG_FEATS]
const_feats = [c for c in NUM_FEATURES if runs[c].nunique(dropna=True) <= 1]
NUM_FEATURES = [c for c in NUM_FEATURES if c not in const_feats]
print(f"{len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical features | dropped constant: {const_feats}")
display(runs.head())

# Run-level outliers in features (IQR, per mode) – reported, not removed: tree models are robust, and these may be real process events
feat_out = {}
for c in NUM_FEATURES:
    s = runs[c]; gq1 = runs.groupby("MODE")[c].transform(lambda x: x.quantile(.25)); gq3 = runs.groupby("MODE")[c].transform(lambda x: x.quantile(.75))
    feat_out[c] = int(((s < gq1 - 3 * (gq3 - gq1)) | (s > gq3 + 3 * (gq3 - gq1))).sum())
fo = pd.Series(feat_out).sort_values(ascending=False)
print("Run-level features with most 'far-out' values (3×IQR within mode):"); display(fo[fo > 0].head(10).to_frame("runs"))

## 16. Target quality & distribution
Labels missing, contradictory (§9), impossible (§7), extreme (§11) – and now: distribution per stage/chamber, multi-modality, and whether train/validation/test targets look alike.

In [ ]:
L = runs[runs[TARGET].notna()].copy()
Lc = L[~L.TARGET_OUTLIER]                         # main analysis set (outliers excluded)
print(f"Labelled runs: {len(L):,} | after excluding flagged outliers: {len(Lc):,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(Lc, x=TARGET, hue="STAGE", bins=70, ax=axes[0], element="step", fill=False)
axes[0].set_title("Removal rate by stage (outliers excluded)")
if "MODE" in Lc:
    order = Lc.groupby("MODE")[TARGET].median().sort_values().index
    sns.stripplot(data=Lc, y="MODE", x=TARGET, order=order, ax=axes[1], size=2.5, alpha=.5, color=PALETTE[0])
    sns.pointplot(data=Lc, y="MODE", x=TARGET, order=order, ax=axes[1], estimator="median", errorbar=None, color=PALETTE[1], markers="D", linestyle="none")
    axes[1].set_title("Removal rate by mode (orange = median)")
save_fig("16_target_distribution")

ks = []
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    x, y = Lc.loc[Lc.SPLIT == a, TARGET], Lc.loc[Lc.SPLIT == b, TARGET]
    if len(x) > 5 and len(y) > 5:
        r = stats.ks_2samp(x, y); ks.append({"pair": f"{a} vs {b}", "KS_stat": r.statistic, "p_value": r.pvalue, "n": f"{len(x)}/{len(y)}"})
print("Do the splits share the same target distribution? (two-sample KS)"); display(pd.DataFrame(ks))

# Multi-modality check per stage: fraction of runs in each mode via a 2-cluster split on the 1-D target (descriptive only)
from sklearn.mixture import GaussianMixture
modes = {}
for s_, d in Lc[Lc.SPLIT == "train"].groupby("STAGE"):
    x = d[[TARGET]].values
    if len(x) > 30:
        bic = [GaussianMixture(k, random_state=RANDOM_STATE).fit(x).bic(x) for k in (1, 2, 3)]
        modes[s_] = int(np.argmin(bic) + 1)
print("Number of target modes per stage suggested by BIC (train only):", modes)

## 18. Univariate EDA – run level
What is typical, how much variation, which categories dominate, where data looks unusual. (Sample-level distributions were shown in §14.)

In [ ]:
key_feats = [c for c in ["duration", "n_samples"] + [f"{c}_start" for c in COUNTERS] +
             [f"{c}_active_mean" for c in proc_cols]][:20]
key_feats = [c for c in key_feats if c in runs.columns and c in NUM_FEATURES]
uni = runs[key_feats].describe(percentiles=[.05, .5, .95]).T
uni["cv_%"] = 100 * uni["std"] / uni["mean"].abs().replace(0, np.nan)
uni["skew"] = runs[key_feats].skew()
display(uni)

ncol = 4; nrow = int(np.ceil(len(key_feats) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.7 * nrow))
for ax, c in zip(axes.ravel(), key_feats):
    sns.histplot(runs[~runs.LONG_RUN.fillna(False).astype(bool)], x=c, hue="MODE", bins=40, ax=ax, element="step", fill=False, legend=(ax is axes.ravel()[0]))
    ax.set_title(c, fontsize=8.5); ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes.ravel()[len(key_feats):]:
    ax.axis("off")
fig.suptitle("Run-level feature distributions by mode (long runs excluded)", y=1.0)
save_fig("18_runlevel_histograms")

for c in [x for x in CAT_FEATURES if x in runs]:
    vc = runs[c].value_counts(normalize=True).mul(100).round(2)
    print(f"{c}: {runs[c].nunique()} levels | dominant = {vc.index[0]} ({vc.iloc[0]}%) | levels with <1% of runs: {(vc < 1).sum()}")

## 19. Bivariate & multivariate EDA
* **Spearman** rank correlation (captures monotonic, non-linear relationships and is robust to outliers) between every feature and the target, **computed within each operating mode** – pooling modes would mostly measure the mode difference (e.g. mode A-123 has both shorter runs *and* ~2× higher removal rate, which creates a strong but spurious pooled duration↔rate correlation).
* p-values corrected for testing many features with **Benjamini–Hochberg FDR**.
* Scatter plots of the strongest relationships, feature–feature correlation (redundancy), and a check for interaction with chamber.

⚠ Correlation ≠ causation: set-points are chosen by recipe, so a feature can correlate with MRR simply because it marks a different recipe or chamber.

In [ ]:
def bh_fdr(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(q, 1); return out

def corr_table(df, feats, by="MODE"):
    rows = []
    for s_, d in df.groupby(by):
        for c in feats:
            x = d[[c, TARGET]].dropna()
            if len(x) > 10 and x[c].nunique() > 1:
                rho, p = stats.spearmanr(x[c], x[TARGET]); r_p = np.corrcoef(x[c], x[TARGET])[0, 1]
                rows.append({by: s_, "feature": c, "spearman": rho, "pearson": r_p, "p": p, "n": len(x)})
    t = pd.DataFrame(rows)
    if len(t):
        t["q_fdr"] = bh_fdr(t.p); t["abs_rho"] = t.spearman.abs()
    return t

LcM = Lc[~Lc.LONG_RUN.fillna(False).astype(bool)]           # long multi-pass runs excluded from relationship analysis
CORR = corr_table(LcM[LcM.SPLIT == "train"], NUM_FEATURES)
top = CORR.sort_values("abs_rho", ascending=False).groupby("MODE").head(8)
print("Strongest associations with removal rate, per mode (train, outliers & long runs excluded):"); display(top.round(4))
pooled = corr_table(LcM[LcM.SPLIT == "train"].assign(_all="all"), ["n_samples", "duration"], by="STAGE") if "n_samples" in NUM_FEATURES else pd.DataFrame()
if len(pooled):
    print("For comparison – the same features pooled by STAGE only (mixing modes) – note how much stronger they look:"); display(pooled[["STAGE", "feature", "spearman", "n"]].round(3))
CORR.to_csv(OUT_DIR / "feature_target_correlations.csv", index=False)

# Heat-strip: top features × stage
topfeats = CORR.sort_values("abs_rho", ascending=False).feature.drop_duplicates().head(15).tolist()
hm = CORR[CORR.feature.isin(topfeats)].pivot(index="feature", columns="MODE", values="spearman").loc[topfeats]
fig, ax = plt.subplots(figsize=(5.5, 0.38 * len(topfeats) + 1))
sns.heatmap(hm, cmap="coolwarm", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "Spearman ρ"})
ax.set_title("Feature ↔ removal rate (within mode)")
save_fig("19_corr_with_target")

# Scatter of the top 4 features
top4 = top.sort_values("abs_rho", ascending=False).feature.drop_duplicates().head(4).tolist()
fig, axes = plt.subplots(1, len(top4), figsize=(4.2 * len(top4), 3.6), squeeze=False)
for ax, c in zip(axes[0], top4):
    sns.scatterplot(data=LcM, x=c, y=TARGET, hue="MODE", s=12, alpha=.6, ax=ax, edgecolor="white", linewidth=.3, legend=(ax is axes[0][0]))
    ax.set_title(c, fontsize=9)
save_fig("19_top_scatter")

In [ ]:
# Multicollinearity among features – many aggregates measure the same thing
fc = LcM[NUM_FEATURES].corr(method="spearman")
pairs = (fc.where(np.triu(np.ones(fc.shape, bool), 1)).stack().rename("rho").reset_index()
           .assign(abs_rho=lambda d: d.rho.abs()).sort_values("abs_rho", ascending=False))
print(f"Feature pairs with |ρ| > 0.95: {(pairs.abs_rho > .95).sum()} (redundant – consider dropping one of each)")
display(pairs.head(10))
sub = [c for c in topfeats if c in fc][:15]
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(fc.loc[sub, sub], cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, square=True, cbar_kws={"label": "Spearman ρ"})
ax.set_title("Correlation among the most target-related features")
save_fig("19_feature_feature_corr")

# Interaction: does the strongest relationship hold in every split of each mode? (stability check)
if top4:
    f0 = top4[0]
    rows_ = []
    for (m_, sp_), d in LcM.groupby(["MODE", "SPLIT"]):
        ok = len(d) > 10 and d[f0].nunique() > 1
        rows_.append({"MODE": m_, "SPLIT": sp_, "n": len(d), "spearman": stats.spearmanr(d[f0], d[TARGET])[0] if ok else np.nan})
    print(f"Relationship of {f0} with removal rate inside each mode and split:"); display(pd.DataFrame(rows_).round(3))

## 20. Segments, trends & comparisons
* Removal rate by **mode** and **machine**, with counts and **bootstrap 95% confidence intervals** for the median.
* Trend across **time** (run start) – drift?
* Trend across **consumable life** (usage counters at run start) – does wear lower removal rate?
* Baseline = overall stage median.

In [ ]:
def boot_ci(x, f=np.median, n=2000, seed=RANDOM_STATE):
    x = np.asarray(x); x = x[~np.isnan(x)]
    if len(x) < 3: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); b = f(rng.choice(x, (n, len(x)), replace=True), axis=1)
    return tuple(np.percentile(b, [2.5, 97.5]))

seg_cols = [c for c in ["MODE", "MACHINE_ID"] + ([] if MACHINE_DATA_REDUNDANT else ["MACHINE_DATA"]) if c in Lc and Lc[c].nunique() > 1]
SEGMENTS = []
for c in seg_cols:
    for lvl, d in Lc.groupby(c):
        lo, hi = boot_ci(d[TARGET].values)
        base = Lc.loc[Lc.STAGE == d.STAGE.mode().iloc[0], TARGET].median()
        SEGMENTS.append({"segment": c, "level": lvl, "n_runs": len(d), "median": d[TARGET].median(), "ci_low": lo, "ci_high": hi,
                         "stage_baseline": base, "diff_vs_baseline_%": 100 * (d[TARGET].median() - base) / base})
SEGMENTS = pd.DataFrame(SEGMENTS)
display(SEGMENTS.round(3))
SEGMENTS.to_csv(OUT_DIR / "segment_comparison.csv", index=False)

if "MODE" in seg_cols:
    s = SEGMENTS[SEGMENTS.segment == "MODE"].sort_values("median")
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(s) + 1.2))
    ax.errorbar(s["median"], s["level"].astype(str), xerr=[s["median"] - s.ci_low, s.ci_high - s["median"]], fmt="o", color=PALETTE[0], ms=8, capsize=3, lw=2)
    for _, r in s.iterrows():
        ax.annotate(f"n={r.n_runs}", (r.ci_high, str(r.level)), xytext=(6, 0), textcoords="offset points", va="center", fontsize=8, color="#52514e")
    ax.set_xlabel(f"median {TARGET} (95% bootstrap CI)"); ax.set_title("Removal rate by mode")
    save_fig("20_segments_ci")

In [ ]:
# Trend over time: runs ordered by start timestamp, rolling median per mode
Lt = Lc.sort_values("t_start")
fig, ax = plt.subplots(figsize=(13, 3.8))
for (s_, d), col in zip(Lt.groupby("MODE"), PALETTE):
    ax.scatter(d.t_start, d[TARGET], s=5, alpha=.25, color=col)
    ax.plot(d.t_start, d[TARGET].rolling(51, center=True, min_periods=10).median(), color=col, lw=2, label=f"{s_} rolling median")
ax.set_xlabel("run start TIMESTAMP"); ax.set_ylabel(TARGET); ax.legend(frameon=False); ax.set_title("Removal rate over time")
save_fig("20_trend_time")

# Consumable life: removal rate vs counter deciles
life = [f"{c}_start" for c in COUNTERS if f"{c}_start" in NUM_FEATURES]
if life:
    ncol = 3; nrow = int(np.ceil(len(life) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.3 * nrow), squeeze=False)
    LIFE_TREND = []
    for ax, c in zip(axes.ravel(), life):
        for (s_, d), col in zip(LcM.groupby("MODE"), PALETTE):
            if d[c].nunique() < 5: continue
            b = pd.qcut(d[c], 10, duplicates="drop")
            agg = d.groupby(b, observed=True).agg(x=(c, "median"), y=(TARGET, "median"), n=(TARGET, "size"))
            ax.plot(agg.x, agg.y, marker="o", ms=5, color=col, lw=2, label=f"{s_}")
            rho = stats.spearmanr(d[c], d[TARGET])[0]
            LIFE_TREND.append({"counter": c, "MODE": s_, "spearman": rho, "first_decile_median": agg.y.iloc[0], "last_decile_median": agg.y.iloc[-1]})
        ax.set_title(c, fontsize=9)
    axes.ravel()[0].legend(frameon=False)
    for ax in axes.ravel()[len(life):]: ax.axis("off")
    fig.suptitle("Median removal rate across consumable-life deciles", y=1.0)
    save_fig("20_consumable_life")
    LIFE_TREND = pd.DataFrame(LIFE_TREND); display(LIFE_TREND.round(3))

    # Replacement events: counter at run start drops vs the previous run on the same chamber
    if "MODE" in runs:
        rr = runs.sort_values("t_start")
        resets = {c: int((rr.groupby(["MODE"])[c].diff() < 0).sum()) for c in life}
        print("Detected consumable replacements (counter decreased between consecutive runs of a chamber):", resets)

## 21. Robustness & trustworthiness
1. **Sensitivity to the outlier decision** – do the top correlations change if flagged labels are kept?
2. **Pearson vs Spearman** – is a relationship driven by a few extreme points?
3. **Uncertainty** – bootstrap CIs (§20) and FDR-corrected q-values (§19).
4. **Practical vs statistical significance** – with ~1,000+ runs tiny ρ can be "significant"; we only highlight |ρ| ≥ 0.3 with q < 0.05.
5. **Representativeness** – does missingness / short runs concentrate in one segment?

In [ ]:
CORR_ALL = corr_table(L[(L.SPLIT == "train") & ~L.LONG_RUN.fillna(False).astype(bool)], NUM_FEATURES)
cmp_ = CORR.merge(CORR_ALL, on=["MODE", "feature"], suffixes=("_clean", "_with_outliers"))
cmp_["delta"] = cmp_.spearman_with_outliers - cmp_.spearman_clean
cmp_["pearson_shift_clean"] = cmp_.pearson_with_outliers - cmp_.pearson_clean
print("Largest changes when outlier labels are kept (Spearman is robust; Pearson is not):")
display(cmp_.reindex(cmp_.pearson_shift_clean.abs().sort_values(ascending=False).index)[["MODE", "feature", "spearman_clean", "spearman_with_outliers", "pearson_clean", "pearson_with_outliers"]].head(8).round(3))

ROBUST = CORR[(CORR.abs_rho >= 0.3) & (CORR.q_fdr < 0.05)].copy()
ROBUST["sign_stable_with_outliers"] = ROBUST.merge(CORR_ALL[["MODE", "feature", "spearman"]], on=["MODE", "feature"], how="left", suffixes=("", "_o")).pipe(lambda d: np.sign(d.spearman) == np.sign(d.spearman_o)).values
print(f"Practically & statistically meaningful associations (|ρ|≥0.3, q<0.05): {len(ROBUST)}")
display(ROBUST.sort_values("abs_rho", ascending=False).head(15).round(4))

# Consistency across splits: is the same association seen in validation?
CORR_VAL = corr_table(LcM[LcM.SPLIT == "validation"], NUM_FEATURES) if (LcM.SPLIT == "validation").sum() > 20 else pd.DataFrame()
if len(CORR_VAL):
    rep = ROBUST.merge(CORR_VAL, on=["MODE", "feature"], suffixes=("_train", "_val"))
    print(f"Of the meaningful train associations, {int((np.sign(rep.spearman_train) == np.sign(rep.spearman_val)).sum())}/{len(rep)} keep their sign in validation; "
          f"median |ρ| train {rep.spearman_train.abs().median():.2f} vs validation {rep.spearman_val.abs().median():.2f}")

if "MODE" in runs:
    q = runs.groupby("MODE").agg(runs=("WAFER_ID", "size"), labelled_pct=(TARGET, lambda s: 100 * s.notna().mean()),
                                         outlier_labels=("TARGET_OUTLIER", "sum"), short_runs=("SHORT_RUN", "sum"), long_runs=("LONG_RUN", "sum"),
                                         zero_pressure_runs=("ZERO_PRESSURE_RUN", "sum"),
                                         mean_missing_frac=("MISSING_SIGNAL_FRAC", "mean"))
    print("Data quality by segment (uneven quality distorts comparisons):"); display(q.round(3))

### 21b. Leakage-safe baseline models (does the preprocessing produce a usable modelling table?)
Pipeline fitted on **train only** (outliers by train-fitted rule excluded), tuned/compared on **validation**, **test** evaluated once. Metrics: MAE (robust, interpretable in target units), RMSE (penalises big misses), R².

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

M = runs[runs[TARGET].notna()].copy()
tr = M[(M.SPLIT == "train") & ~M.OUTLIER_TRAIN_RULE & ~M.LONG_RUN.fillna(False).astype(bool)].copy()   # clean training set
va = M[M.SPLIT == "validation"].copy(); te = M[M.SPLIT == "test"].copy()                                    # evaluated as-is
cat_in = [c for c in CAT_FEATURES if c in M and M[c].nunique() > 1]
for c in cat_in:
    for d in (tr, va, te): d[c] = d[c].astype(str)
X_cols = NUM_FEATURES + cat_in

def make_pre(scale):
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True))] + ([("sc", StandardScaler())] if scale else []))
    return ColumnTransformer([("num", num, NUM_FEATURES),
                              ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_in)])

models = {
    "Baseline: train median per mode": None,
    "Ridge (scaled)": Pipeline([("pre", make_pre(True)), ("m", RidgeCV(alphas=np.logspace(-3, 3, 13)))]),
    "Random forest": Pipeline([("pre", make_pre(False)), ("m", RandomForestRegressor(400, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE))]),
    "HistGradientBoosting": Pipeline([("pre", make_pre(False)), ("m", HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=RANDOM_STATE))]),
}
def metrics(y, p): return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5, "R2": r2_score(y, p)}

seg = "MODE" if "MODE" in M else "STAGE"
RESULTS, fitted = [], {}
gkf = GroupKFold(n_splits=5)
for name, mdl in models.items():
    if mdl is None:
        med = tr.groupby(seg)[TARGET].median(); fallback = tr[TARGET].median()
        pv = va[seg].astype(str).map(med.rename(index=str)).fillna(fallback)
        cv = np.nan
    else:
        cv = -cross_val_score(mdl, tr[X_cols], tr[TARGET], groups=tr.WAFER_ID, cv=gkf, scoring="neg_mean_absolute_error").mean()
        mdl.fit(tr[X_cols], tr[TARGET]); fitted[name] = mdl; pv = mdl.predict(va[X_cols])
    row = {"model": name, "CV_MAE_train(GroupKFold)": cv}
    if len(va):
        ok = ~va.OUTLIER_TRAIN_RULE.values
        row.update({f"val_{k}": v for k, v in metrics(va[TARGET][ok], np.asarray(pv)[ok]).items()})        # flagged labels excluded
        row.update({f"val_all_{k}": v for k, v in metrics(va[TARGET], pv).items()})                       # sensitivity: all labels
    RESULTS.append(row)
RESULTS = pd.DataFrame(RESULTS); display(RESULTS.round(3))
print("val_* = validation without flagged labels (used for selection); val_all_* = including them (sensitivity).")

best = RESULTS.dropna(subset=["CV_MAE_train(GroupKFold)"]).sort_values("val_MAE" if "val_MAE" in RESULTS else "CV_MAE_train(GroupKFold)").model.iloc[0]
print("Selected on validation:", best)
if len(te):
    pt = fitted[best].predict(te[X_cols])
    TEST_METRICS = metrics(te[TARGET], pt)
    TEST_METRICS_CLEAN = metrics(te.loc[~te.OUTLIER_TRAIN_RULE, TARGET], pt[~te.OUTLIER_TRAIN_RULE.values])
    print("TEST (evaluated once):", {k: round(v, 3) for k, v in TEST_METRICS.items()}, "| excluding flagged test labels:", {k: round(v, 3) for k, v in TEST_METRICS_CLEAN.items()})
RESULTS.to_csv(OUT_DIR / "model_metrics.csv", index=False)

if len(va):
    pi = permutation_importance(fitted[best], va[X_cols], va[TARGET], n_repeats=5, random_state=RANDOM_STATE, scoring="neg_mean_absolute_error", n_jobs=-1)
    IMP = pd.Series(pi.importances_mean, index=X_cols).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(IMP.index[::-1], IMP.values[::-1], color=PALETTE[0], height=0.6)
    ax.set_xlabel("increase in validation MAE when shuffled"); ax.set_title(f"Permutation importance – {best}")
    save_fig("21_permutation_importance")

## 22. Insights – finding → evidence → meaning → caveat → next step
The cell below **writes the insights from the numbers this run produced** (nothing is hard-coded), so they stay correct if the data or a cleaning rule changes. Edit/extend the wording in the markdown cell after it for your final report.

In [ ]:
INS = []
def add(topic, finding, evidence, meaning, caveat, next_step):
    INS.append({"topic": topic, "finding": finding, "evidence": evidence, "meaning": meaning, "caveat": caveat, "next_step": next_step})

# 1. Data quality
n_rules = int((RULES.violations > 0).sum())
add("Data quality",
    f"{len(RAW_SENSOR):,} sensor samples → {len(runs):,} runs; {n_rules} validation rule(s) had violations; {len(CLEAN_LOG)} logged cleaning actions.",
    "§4–§9 tables, cleaning_log.csv",
    "Raw data is largely structurally clean; the main risks are at the label level (outliers, missing labels) rather than in sensors.",
    "Field meanings and units are undocumented, so plausibility rules beyond sign and ordering cannot be checked.",
    "Confirm units/ranges with the data owner if this were a production project.")

# 2. Target outliers
if n_out:
    ex_vals = labels.loc[labels.TARGET_OUTLIER, TARGET]
    add("Label errors",
        f"{n_out} labelled runs have removal rates far outside their operating mode's distribution (|robust z|>{Z_LIMIT} within mode); values {ex_vals.min():,.1f}–{ex_vals.max():,.1f} vs mode medians {labels.groupby(grp)[TARGET].median().round(1).to_dict() if grp else round(labels[TARGET].median(), 1)}.",
        "§11 box plots, outlier table",
        "Left in, they dominate squared-error metrics and Pearson correlations (see §21), distorting both models and EDA.",
        "They could be real extreme events, but values orders of magnitude off are much more consistent with recording/unit errors.",
        "Exclude from training, report test metrics with and without them, and ask the organisers/metrology team.")

# 3. Modes
if "MODE" in SEGMENTS.get("segment", pd.Series(dtype=str)).values:
    s = SEGMENTS[SEGMENTS.segment == "MODE"].sort_values("median")
    add("Operating modes",
        f"Median removal rate ranges from {s['median'].min():,.1f} ({s.level.iloc[0]}) to {s['median'].max():,.1f} ({s.level.iloc[-1]}) across mode segments; BIC suggests {modes} target modes per stage.",
        "§16 strip plot, §20 CI plot (non-overlapping CIs = robust difference)",
        "Stage/chamber is the first-order driver of removal rate; any analysis that pools them mostly measures the mode difference (Simpson's paradox risk).",
        "Chamber may proxy for recipe or wafer type, which is not recorded – it does not prove the chamber itself causes the difference.",
        "Analyse and model within mode (or include mode as a feature) – this matches the 'two/three mode division' experiments in the repo.")

# 4. Strongest associations
if len(ROBUST):
    r0 = ROBUST.sort_values("abs_rho", ascending=False).iloc[0]
    add("Process drivers",
        f"{len(ROBUST)} feature–mode pairs show |ρ|≥0.3 with q<0.05; strongest: {r0.feature} in mode {r0.MODE} (ρ={r0.spearman:.2f}, n={int(r0.n)}).",
        "§19 heat-strip, scatter plots; feature_target_correlations.csv",
        "These signals carry real predictive information about removal rate and are the first candidates for monitoring.",
        "Associations, not causes – set-points differ by recipe; many aggregates are highly collinear (§19), so they can share the same signal.",
        "Confirm the top relationships hold in every split (§19 table) and keep one feature from each redundant cluster.")

# 5. Consumable wear
if life and len(LIFE_TREND):
    lt = LIFE_TREND.reindex(LIFE_TREND.spearman.abs().sort_values(ascending=False).index).iloc[0]
    add("Consumable wear",
        f"Removal rate changes most with {lt.counter} in mode {lt.MODE} (ρ={lt.spearman:.2f}; median {lt.first_decile_median:,.1f} in first life decile vs {lt.last_decile_median:,.1f} in last).",
        "§20 consumable-life plot",
        "Consumable age is an actionable lever: replacement/conditioning schedules may stabilise removal rate.",
        "Wear is confounded with calendar time and anything else that changed over the period; several counters move in lock-step (ρ≈1), so they cannot be separated.",
        "Model removal rate vs wear within replacement cycles; test whether rate recovers after replacement events.")

# 5b. Hidden missing data: zero-pressure runs
if ZERO_PRESSURE_RUN.any():
    nz = int(ZERO_PRESSURE_RUN.sum()); zmodes = zp[zp.ZERO_PRESSURE_RUN].MODE.value_counts()
    med = cmp_zero["median"]
    add("Hidden missing data",
        f"{nz} runs ({100*nz/len(runs):.0f}% of all runs) report every pressure as exactly 0 for the whole run; they occur in {zmodes.to_dict()}.",
        "§8 zero-run tables: removal rate of these runs vs. other runs in the same mode: " + "; ".join(f"{k[0]} zero={k[1]}: {v:,.1f}" for k, v in med.items()),
        "Similar removal rates show the wafers were polished normally, so the zeros mean 'not recorded'. Treated as real zeros they would create a fake 'zero pressure → normal rate' pattern.",
        "We cannot recover the true pressures; within that mode, pressure-based conclusions rest on fewer runs.",
        "Keep the ZERO_PRESSURE_RUN indicator, never treat these zeros as values, and ask why logging was off for that mode.")

# 5c. Long multi-pass runs
if ts.LONG_RUN.any():
    lr = ts[ts.LONG_RUN]
    add("Abnormal runs",
        f"{len(lr)} runs last more than twice their mode's median duration (up to {lr.duration.max():,.0f} vs typical {ts.duration.median():,.0f} time units; up to {int(lr.n_samples.max()):,} samples).",
        "§17 long-run table and duration plots",
        "These IDs probably hold several polishing passes (rework or a reused wafer ID) but one averaged label, so their per-run features are not comparable with normal runs.",
        "Without process logs it cannot be confirmed whether they are rework or ID reuse.",
        "Excluded from model training and from correlation analysis; split them into passes if the start of each pass can be detected.")

# 5d. Same physical wafer in several splits
if len(cross):
    add("Split leakage risk",
        f"{len(cross)} wafer IDs appear in more than one organiser split (typically stage A in one split, stage B in another); {int(stages_per_wafer.get(2, 0))} wafers have both stages in the same split.",
        "§9 cross-split table",
        "The same wafer's material/incoming thickness can link a training run to a test run, so test scores may be slightly optimistic for 'new wafer' use.",
        "Effect size is unknown; the stages are different operations with different labels.",
        "For a stricter estimate, re-split by WAFER_ID (GroupKFold already does this inside train).")

# 6. Split comparability
if len(ks):
    kk = pd.DataFrame(ks)
    add("Split comparability",
        "Target distributions across splits: " + "; ".join(f"{r.pair} KS={r.KS_stat:.3f}" for r in kk.itertuples()) + ".",
        "§16 KS table",
        ("All KS statistics are small (<0.1): validation/test look like the same population as training, so validation scores should transfer."
         if kk.KS_stat.max() < 0.1 else
         "At least one pair differs noticeably (KS ≥ 0.1): check whether segment mix differs between splits (§13) before trusting validation scores."),
        "A KS test only checks the target's marginal distribution, not feature relationships.",
        "Also monitor feature drift if new data arrive.")

# 7. Model readiness
if len(RESULTS) and "val_MAE" in RESULTS:
    b = RESULTS.set_index("model")
    base_mae = b.loc["Baseline: train median per mode", "val_MAE"]; best_mae = b.loc[best, "val_MAE"]
    add("Predictability",
        f"Best model ({best}) validation MAE {best_mae:,.2f} vs {base_mae:,.2f} for a per-mode median baseline ({100*(best_mae/base_mae-1):+.0f}% vs baseline).",
        "§21b model table, permutation importance",
        ("The sensor features add predictive value beyond knowing the operating mode." if best_mae < 0.95 * base_mae else
         "Sensor features add little beyond the operating mode with these default models – most explainable variance is the mode itself."),
        "Default hyper-parameters; neighbouring-wafer features (known winners in this challenge) were not used.",
        "Tune on validation, try per-mode models and time-aware neighbour features with strict 'earlier-only' rules.")

INSIGHTS = pd.DataFrame(INS)
for i, r in INSIGHTS.iterrows():
    md(f"**{i+1}. {r.topic}** — {r.finding}  \n*Evidence:* {r.evidence}  \n*Meaning:* {r.meaning}  \n*Caveat:* {r.caveat}  \n*Next step:* {r.next_step}")
INSIGHTS.to_csv(OUT_DIR / "insights.csv", index=False)

### Your written insights (edit after running)
Use the numbers printed above. Keep observation and explanation apart.

| Level | Your text |
|---|---|
| Observation | … |
| Deeper finding | … |
| Interpretation | … |
| Limitation | … |
| Suggested next step | … |

**Recommendations & limitations** – summarise: (1) which runs/labels to trust, (2) which variables matter within each mode, (3) what to monitor, (4) what the data cannot tell us (units, recipes, causal effects).

## 23. Final validation & before/after comparison
Re-run the profile and the rules on the cleaned data, reconcile row counts, and compare distributions before vs after cleaning.

In [ ]:
PROFILE_AFTER = pd.concat([profile(sensor.drop(columns=["DT"], errors="ignore"), "sensor"), profile(labels, "labels"), profile(runs, "runs")], ignore_index=True)
PROFILE_AFTER.to_csv(OUT_DIR / "quality_profile_after.csv", index=False)

final_checks = {
    "sensor: negative counters/process values": int(sum((sensor[c] < 0).sum() for c in nonneg_cols)),
    "sensor: timestamps decreasing within run": int((sensor.groupby(RUN_KEY, dropna=False)["TIMESTAMP"].diff() < 0).sum()),
    "sensor: exact duplicate rows": int(sensor.duplicated(content_cols).sum()),
    "sensor: duplicate (run, TIMESTAMP)": int(sensor.duplicated(RUN_KEY + ["TIMESTAMP"]).sum()),
    "labels: duplicate run keys": int(labels.duplicated(lab_key).sum()),
    "labels: non-positive target": int((labels[TARGET] <= 0).sum()),
    "runs: duplicate run keys": int(runs.duplicated(RUN_KEY).sum()),
    "runs: rows multiplied by join": int(len(runs) - len(ts)),
    "runs: target-derived columns in FEATURES": len(set(FEATURES) & {TARGET, "TARGET_OUTLIER", "OUTLIER_TRAIN_RULE", "TARGET_ROBUST_Z"}),
}
FINAL = pd.DataFrame({"check": list(final_checks), "remaining_issues": list(final_checks.values())})
FINAL["status"] = np.where(FINAL.remaining_issues == 0, "PASS", "REVIEW")
display(FINAL)

recon = pd.DataFrame({
    "stage": ["raw sensor rows", "clean sensor rows", "runs", "raw label rows", "clean label rows", "runs with label", "runs used for training (train, non-outlier)"],
    "count": [len(RAW_SENSOR), len(sensor), len(runs), len(RAW_LABELS), len(labels), int(runs[TARGET].notna().sum()), len(tr)]})
display(recon)

# Before/after: missingness and distributions of signals
cmp_rows = []
for c in NUMERIC_SIGNALS:
    a = pd.to_numeric(RAW_SENSOR[c], errors="coerce"); b = sensor[c]
    ks_ = stats.ks_2samp(a.dropna().sample(min(50000, a.notna().sum()), random_state=1), b.dropna().sample(min(50000, b.notna().sum()), random_state=1)).statistic if a.notna().any() and b.notna().any() else np.nan
    cmp_rows.append({"column": c, "missing_before": int(a.isna().sum()), "missing_after": int(b.isna().sum()),
                     "mean_before": a.mean(), "mean_after": b.mean(), "KS_before_vs_after": ks_})
BEFORE_AFTER = pd.DataFrame(cmp_rows); display(BEFORE_AFTER.round(4))
print("KS ≈ 0 ⇒ cleaning did not materially change the signal distributions.")

t_before = RAW_LABELS[TARGET].pipe(pd.to_numeric, errors="coerce")
print(f"Target mean before {t_before.mean():,.2f} / median {t_before.median():,.2f}  →  after excluding flagged outliers mean {labels.loc[~labels.TARGET_OUTLIER, TARGET].mean():,.2f} / median {labels.loc[~labels.TARGET_OUTLIER, TARGET].median():,.2f}")
if "MODE" in runs:
    aff = runs.groupby("MODE")["TARGET_OUTLIER"].agg(["sum", "size"]).assign(pct=lambda d: 100 * d["sum"] / d["size"])
    print("Were some segments disproportionately affected by outlier exclusion?"); display(aff.round(2))

## 24. Deliverables

| File (in `outputs/`) | Content |
|---|---|
| `clean_sensor_samples.csv.gz` | cleaned sample-level data (types fixed, duplicates resolved, short gaps interpolated) |
| `run_level_features_labeled.csv` | analysis/modelling-ready table: one row per run + target + outlier flags |
| `data_dictionary.csv` | columns, roles, units + derived-feature formulas |
| `cleaning_log.csv` | every change, reason, records affected, method |
| `file_manifest.csv`, `run_metadata.json` | raw files, checksums, versions – reproducibility |
| `quality_profile_before/after.csv`, `validation_rules.csv`, `final_checks.csv`, `before_after.csv` | quality report |
| `feature_target_correlations.csv`, `segment_comparison.csv`, `figures/*.png` | EDA report material |
| `insights.csv`, `model_metrics.csv` | insights + baseline modelling results |

In [ ]:
derived = pd.DataFrame([
    {"column": "<counter>_start", "role": "feature", "meaning": "first value of the counter in the run (consumable age)", "units": "as counter", "expected_values": ">= 0", "present": True},
    {"column": "<counter>_delta", "role": "feature", "meaning": "last - first counter value in run", "units": "as counter", "expected_values": ">= 0", "present": True},
    {"column": "<signal>_mean/_std/_p10/_p50/_p90", "role": "feature", "meaning": "per-run statistics of process signal", "units": "as signal", "expected_values": "", "present": True},
    {"column": "<signal>_active_mean", "role": "feature", "meaning": "mean while HEAD_ROTATION > 0 (fallback: overall mean)", "units": "as signal", "expected_values": "", "present": True},
    {"column": "TOTAL_SLURRY_FLOW_mean", "role": "feature", "meaning": "sum of slurry line A+B+C means", "units": "flow", "expected_values": ">= 0", "present": True},
    {"column": "HEAD_STAGE_ROT_RATIO", "role": "feature", "meaning": "HEAD_ROTATION_mean / STAGE_ROTATION_mean (NaN if denominator 0)", "units": "ratio", "expected_values": ">= 0", "present": True},
    {"column": "n_samples, duration, median_dt, n_gaps", "role": "feature", "meaning": "run timing / completeness (gap = interval > 3x run median)", "units": "count / time", "expected_values": "> 0", "present": True},
    {"column": "MISSING_SIGNAL_FRAC", "role": "feature", "meaning": "share of samples with any missing signal before interpolation", "units": "fraction", "expected_values": "[0,1]", "present": True},
    {"column": "CHAMBER_GROUP / MODE", "role": "category", "meaning": "chamber group of the run (123 or 456, from all its samples); MODE = STAGE-CHAMBER_GROUP", "units": "-", "expected_values": "A-123, A-456, B-456 (+ mixed)", "present": True},
    {"column": "LONG_RUN", "role": "flag", "meaning": "duration > 2x median duration of its mode (likely several passes)", "units": "-", "expected_values": "bool", "present": True},
    {"column": "ZERO_PRESSURE_RUN", "role": "feature/flag", "meaning": "all pressure signals 0 for the whole run (= not recorded); those aggregates set to NaN", "units": "-", "expected_values": "0/1", "present": True},
    {"column": "TARGET_ROBUST_Z / TARGET_OUTLIER / OUTLIER_TRAIN_RULE", "role": "flag (never a feature)", "meaning": f"median/MAD z of target within MODE (fallback stage); |z|>{Z_LIMIT}; same rule with train-only thresholds", "units": "-", "expected_values": "", "present": True},
])
pd.concat([dd, derived], ignore_index=True).to_csv(OUT_DIR / "data_dictionary.csv", index=False)

sensor.drop(columns=["DT"], errors="ignore").to_csv(OUT_DIR / "clean_sensor_samples.csv.gz", index=False, compression="gzip")
runs.to_csv(OUT_DIR / "run_level_features_labeled.csv", index=False)
pd.DataFrame(CLEAN_LOG).to_csv(OUT_DIR / "cleaning_log.csv", index=False)
FINAL.to_csv(OUT_DIR / "final_checks.csv", index=False)
BEFORE_AFTER.to_csv(OUT_DIR / "before_after.csv", index=False)
json.dump({"source_url": DATA_URL, "zip_sha256": ZIP_SHA256, "extracted_at": EXTRACTED_AT, "run_finished": dt.datetime.now().isoformat(timespec="seconds"),
           "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "random_state": RANDOM_STATE,
           "z_limit": Z_LIMIT, "max_interp": MAX_INTERP}, open(OUT_DIR / "run_metadata.json", "w"), indent=2)

print("Cleaning log:"); display(pd.DataFrame(CLEAN_LOG))
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file(): print(f"{p.relative_to(OUT_DIR)}  ({p.stat().st_size/1024:,.0f} KB)")

In [ ]:
# Zip everything and download (Colab)
archive = shutil.make_archive(str(WORK_DIR / "phm_preprocessing_eda_outputs"), "zip", OUT_DIR)
print("Created", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)